In [18]:
# Protein Scaffold Datasets = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets"
# de_novo_sequence_CAH2 = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/de_novo_sequence_CAH2.txt"
# mab_target_sequence= "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_target_sequence.txt"
# mab_training_sequence= "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_target_sequence.txt"
# p5A_training_sequence = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/p5A_training_sequence.txt"
# target_sequence_CAH2 = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/target_sequence_CAH2.txt"
# training_sequences_CAH2 = "training_sequences_CAH2"

In [44]:
# ============================================================
# HYBRID PROTEIN SCAFFOLD GAP FILLING SYSTEM
# One-cell Kaggle version
#
# Includes:
# 1. Dataset loading
# 2. FASTA parsing
# 3. 11-mer masked ML dataset
# 4. Base 8 ML models
# 5. Raw + Row Average + SVD features
# 6. Best-model weighted ensemble
# 7. Known-size gap filling
# 8. Known-mass probabilistic candidate generation
# 9. Hybrid reranking
# 10. Evaluation + saving outputs
# ============================================================

import os
import re
import math
import time
import random
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

FAST_MODE = True

if FAST_MODE:
    MAX_ML_SAMPLES = 30000
    SAMPLES_PER_SEQUENCE = 2
    RF_TREES = 100
    ET_TREES = 100
    MLP_MAX_ITER = 150
else:
    MAX_ML_SAMPLES = None
    SAMPLES_PER_SEQUENCE = 5
    RF_TREES = 200
    ET_TREES = 200
    MLP_MAX_ITER = 300

TOP_N_MODELS = 5


# ============================================================
# 2. DATASET PATHS
# ============================================================

DATA_DIR = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets"

DE_NOVO_CAH2 = f"{DATA_DIR}/de_novo_sequence_CAH2.txt"
MAB_TARGET = f"{DATA_DIR}/mab_target_sequence.txt"
MAB_TRAINING = f"{DATA_DIR}/mab_training_sequence.txt"
P5A_TRAINING = f"{DATA_DIR}/p5A_training_sequence.txt"
CAH2_TARGET = f"{DATA_DIR}/target_sequence_CAH2.txt"
CAH2_TRAINING = f"{DATA_DIR}/training_sequences_CAH2.txt"


def find_file(filename, root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None


paths = {
    "DE_NOVO_CAH2": ("de_novo_sequence_CAH2.txt", DE_NOVO_CAH2),
    "MAB_TARGET": ("mab_target_sequence.txt", MAB_TARGET),
    "MAB_TRAINING": ("mab_training_sequence.txt", MAB_TRAINING),
    "P5A_TRAINING": ("p5A_training_sequence.txt", P5A_TRAINING),
    "CAH2_TARGET": ("target_sequence_CAH2.txt", CAH2_TARGET),
    "CAH2_TRAINING": ("training_sequences_CAH2.txt", CAH2_TRAINING),
}

fixed_paths = {}

for key, (fname, path) in paths.items():
    if os.path.exists(path):
        fixed_paths[key] = path
    else:
        fixed_paths[key] = find_file(fname)

DE_NOVO_CAH2 = fixed_paths["DE_NOVO_CAH2"]
MAB_TARGET = fixed_paths["MAB_TARGET"]
MAB_TRAINING = fixed_paths["MAB_TRAINING"]
P5A_TRAINING = fixed_paths["P5A_TRAINING"]
CAH2_TARGET = fixed_paths["CAH2_TARGET"]
CAH2_TRAINING = fixed_paths["CAH2_TRAINING"]

print("==== Dataset Paths ====")
for name, path in fixed_paths.items():
    print(name, "=>", path, "| exists:", path is not None and os.path.exists(path))


# ============================================================
# 3. AMINO ACID VOCABULARY AND MASS TABLE
# ============================================================

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
GAP_TOKEN = "-"

AA_MASS = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99,
    "T": 101, "C": 103, "L": 113, "I": 113, "N": 114,
    "D": 115, "Q": 128, "K": 128, "E": 129, "M": 131,
    "H": 137, "F": 147, "R": 156, "Y": 163, "W": 186
}

aa_to_int = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa_to_int[GAP_TOKEN] = 0

int_to_aa = {v: k for k, v in aa_to_int.items()}


def peptide_mass(seq):
    return int(sum(AA_MASS.get(aa, 0) for aa in seq))


def is_valid_protein_sequence(seq):
    return all(ch in AMINO_ACIDS for ch in seq)


def encode_sequence(seq):
    return [aa_to_int.get(ch, 0) for ch in seq]


print("\n==== Amino Acid Checks ====")
print("Mass GPEH:", peptide_mass("GPEH"))
print("Mass SVSYDQA:", peptide_mass("SVSYDQA"))


# ============================================================
# 4. FASTA / TEXT PARSER
# ============================================================

def read_text(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def clean_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    return seq


def parse_fasta_or_sequences(path):
    text = read_text(path)
    sequences = []
    current = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if current:
                seq = clean_sequence("".join(current))
                if seq:
                    sequences.append(seq)
                current = []
        else:
            current.append(line)

    if current:
        seq = clean_sequence("".join(current))
        if seq:
            sequences.append(seq)

    return sequences


mab_target_seq = parse_fasta_or_sequences(MAB_TARGET)[0]
mab_train_seqs = parse_fasta_or_sequences(MAB_TRAINING)

p5a_train_seqs = parse_fasta_or_sequences(P5A_TRAINING)

cah2_target_seq = parse_fasta_or_sequences(CAH2_TARGET)[0]
cah2_train_seqs = parse_fasta_or_sequences(CAH2_TRAINING)

all_homologs = mab_train_seqs + p5a_train_seqs + cah2_train_seqs

print("\n==== Loaded Sequences ====")
print("Mab target length:", len(mab_target_seq))
print("Mab training sequences:", len(mab_train_seqs))
print("P5A training sequences:", len(p5a_train_seqs))
print("CAH2 target length:", len(cah2_target_seq))
print("CAH2 training sequences:", len(cah2_train_seqs))
print("All homolog sequences:", len(all_homologs))


# ============================================================
# 5. GENERATE 11-MER ML DATASET
# ============================================================

def generate_masked_kmer_dataset(sequences, k=11, max_samples=None):
    X = []
    y_letters = []

    for seq in sequences:
        if len(seq) < k:
            continue

        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]

            if not is_valid_protein_sequence(kmer):
                continue

            masked_first = GAP_TOKEN + kmer[1:]
            X.append(encode_sequence(masked_first))
            y_letters.append(kmer[0])

            masked_last = kmer[:-1] + GAP_TOKEN
            X.append(encode_sequence(masked_last))
            y_letters.append(kmer[-1])

            if max_samples is not None and len(X) >= max_samples:
                return np.array(X), np.array(y_letters)

    return np.array(X), np.array(y_letters)


X_ml, y_ml_letters = generate_masked_kmer_dataset(
    all_homologs,
    k=11,
    max_samples=MAX_ML_SAMPLES
)

label_encoder = LabelEncoder()
y_ml = label_encoder.fit_transform(y_ml_letters)

X_train_ml, X_val_ml, y_train_ml, y_val_ml = train_test_split(
    X_ml,
    y_ml,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_ml
)

print("\n==== ML Dataset ====")
print("X_ml:", X_ml.shape)
print("y_ml:", y_ml.shape)
print("Classes:", list(label_encoder.classes_))
print("Train:", X_train_ml.shape)
print("Validation:", X_val_ml.shape)


# ============================================================
# 6. FEATURE FUNCTIONS
# ============================================================

def row_average_features(X):
    return X.mean(axis=1).reshape(-1, 1)


X_train_row = row_average_features(X_train_ml)
X_val_row = row_average_features(X_val_ml)

print("\n==== Feature Shapes ====")
print("Raw feature shape:", X_train_ml.shape)
print("Row-average shape:", X_train_row.shape)


# ============================================================
# 7. BASE 8 MODELS
# ============================================================

def get_base_8_models():
    models = {
        "kNN": KNeighborsClassifier(
            n_neighbors=5
        ),

        "DecisionTree": DecisionTreeClassifier(
            max_depth=20,
            min_samples_split=5,
            random_state=RANDOM_STATE
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=RF_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=ET_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        ),

        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000,
                n_jobs=-1,
                random_state=RANDOM_STATE
            ))
        ]),

        "LinearSVC": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                random_state=RANDOM_STATE
            ))
        ]),

        "MLP": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(150, 75),
                activation="tanh",
                max_iter=MLP_MAX_ITER,
                random_state=RANDOM_STATE
            ))
        ])
    }

    return models


print("\n==== Base 8 Models ====")
for name in get_base_8_models():
    print("-", name)


# ============================================================
# 8. TRAIN/EVALUATE HELPERS
# ============================================================

def safe_predict_proba(model, X, num_classes):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)

        if scores.ndim == 1:
            scores = np.vstack([-scores, scores]).T

        scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    preds = model.predict(X)
    proba = np.zeros((len(preds), num_classes))

    for i, p in enumerate(preds):
        proba[i, int(p)] = 1.0

    return proba


def train_and_evaluate_models(models, X_train, X_val, y_train, y_val, feature_type="raw"):
    results = []
    fitted_models = {}

    for name, model in models.items():
        print(f"\nTraining {feature_type}__{name}...")

        start_time = time.time()

        try:
            model.fit(X_train, y_train)

            train_pred = model.predict(X_train)
            val_pred = model.predict(X_val)

            train_acc = accuracy_score(y_train, train_pred)
            val_acc = accuracy_score(y_val, val_pred)

            elapsed = time.time() - start_time

            model_key = f"{feature_type}__{name}"

            results.append({
                "model_name": model_key,
                "base_model": name,
                "feature_type": feature_type,
                "train_accuracy": train_acc,
                "validation_accuracy": val_acc,
                "train_time_sec": elapsed
            })

            fitted_models[model_key] = model

            print(f"{model_key}")
            print(f"Train accuracy: {train_acc:.4f}")
            print(f"Validation accuracy: {val_acc:.4f}")
            print(f"Time: {elapsed:.2f} sec")

        except Exception as e:
            print(f"{feature_type}__{name} failed: {e}")

    results_df = pd.DataFrame(results)

    if len(results_df) > 0:
        results_df = results_df.sort_values(
            "validation_accuracy",
            ascending=False
        ).reset_index(drop=True)

    return results_df, fitted_models


# ============================================================
# 9. TRAIN BASE 8 ON RAW, ROW AVERAGE, AND SVD FEATURES
# ============================================================

print("\n\n==============================")
print("Training RAW models")
print("==============================")

base8_raw_results, base8_raw_fitted = train_and_evaluate_models(
    models=get_base_8_models(),
    X_train=X_train_ml,
    X_val=X_val_ml,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="raw"
)

print("\n\n==============================")
print("Training ROW AVERAGE models")
print("==============================")

base8_row_results, base8_row_fitted = train_and_evaluate_models(
    models=get_base_8_models(),
    X_train=X_train_row,
    X_val=X_val_row,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="row_average"
)

print("\n\n==============================")
print("Training SVD models")
print("==============================")

base_8_svd_models = {}

for name, model in get_base_8_models().items():
    base_8_svd_models[name] = Pipeline([
        ("svd", TruncatedSVD(n_components=5, random_state=RANDOM_STATE)),
        ("model", model)
    ])

base8_svd_results, base8_svd_fitted = train_and_evaluate_models(
    models=base_8_svd_models,
    X_train=X_train_ml,
    X_val=X_val_ml,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="svd"
)


# ============================================================
# 10. COMBINE RESULTS
# ============================================================

base8_all_results = pd.concat([
    base8_raw_results,
    base8_row_results,
    base8_svd_results
], ignore_index=True)

base8_all_results = base8_all_results.sort_values(
    "validation_accuracy",
    ascending=False
).reset_index(drop=True)

base8_all_fitted = {}
base8_all_fitted.update(base8_raw_fitted)
base8_all_fitted.update(base8_row_fitted)
base8_all_fitted.update(base8_svd_fitted)

print("\n==== All Model Results ====")
print("Total trained models:", len(base8_all_fitted))
display(base8_all_results)

top_base8_models_df = base8_all_results.head(TOP_N_MODELS).copy()

print("\n==== Selected Top Models ====")
display(top_base8_models_df)


# ============================================================
# 11. WEIGHTED ENSEMBLE
# ============================================================

class Base8ModelWrapper:
    def __init__(self, model_name, model, feature_type, validation_accuracy):
        self.model_name = model_name
        self.model = model
        self.feature_type = feature_type
        self.validation_accuracy = validation_accuracy

    def transform(self, X):
        if self.feature_type == "raw":
            return X

        if self.feature_type == "row_average":
            return row_average_features(X)

        if self.feature_type == "svd":
            return X

        raise ValueError(f"Unknown feature type: {self.feature_type}")

    def predict(self, X):
        Xt = self.transform(X)
        return self.model.predict(Xt)

    def predict_proba(self, X):
        Xt = self.transform(X)
        return safe_predict_proba(
            self.model,
            Xt,
            num_classes=len(label_encoder.classes_)
        )


top_base8_wrapped_models = []

for _, row in top_base8_models_df.iterrows():
    model_name = row["model_name"]
    feature_type = row["feature_type"]
    model = base8_all_fitted[model_name]

    wrapped = Base8ModelWrapper(
        model_name=model_name,
        model=model,
        feature_type=feature_type,
        validation_accuracy=row["validation_accuracy"]
    )

    top_base8_wrapped_models.append(wrapped)

print("\n==== Models Used in Ensemble ====")
for model in top_base8_wrapped_models:
    print(model.model_name, "| acc:", model.validation_accuracy)


def weighted_ensemble_predict_proba(wrapped_models, X):
    weights = np.array([m.validation_accuracy for m in wrapped_models])
    weights = weights / weights.sum()

    final_proba = None

    for model, weight in zip(wrapped_models, weights):
        proba = model.predict_proba(X)

        if final_proba is None:
            final_proba = weight * proba
        else:
            final_proba += weight * proba

    return final_proba


def weighted_ensemble_predict(wrapped_models, X):
    proba = weighted_ensemble_predict_proba(wrapped_models, X)
    return np.argmax(proba, axis=1)


ensemble_val_pred = weighted_ensemble_predict(
    top_base8_wrapped_models,
    X_val_ml
)

ensemble_val_acc = accuracy_score(y_val_ml, ensemble_val_pred)

print("\n==== Ensemble Validation ====")
print("Best single model accuracy:", base8_all_results.iloc[0]["validation_accuracy"])
print("Weighted ensemble accuracy:", ensemble_val_acc)


def top_k_accuracy(y_true, proba, k=3):
    top_k = np.argsort(proba, axis=1)[:, -k:]
    correct = 0

    for true_label, candidates in zip(y_true, top_k):
        if true_label in candidates:
            correct += 1

    return correct / len(y_true)


ensemble_val_proba = weighted_ensemble_predict_proba(
    top_base8_wrapped_models,
    X_val_ml
)

print("\n==== Ensemble Top-k Accuracy ====")
for k in [1, 2, 3, 5]:
    acc = top_k_accuracy(y_val_ml, ensemble_val_proba, k=k)
    print(f"Top-{k} accuracy: {acc:.4f}")


# ============================================================
# 12. KNOWN-SIZE GAP PREDICTION
# ============================================================

def predict_masked_kmer(masked_kmer):
    x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)

    proba = weighted_ensemble_predict_proba(
        top_base8_wrapped_models,
        x
    )[0]

    pred_id = int(np.argmax(proba))
    pred_aa = label_encoder.inverse_transform([pred_id])[0]
    confidence = float(proba[pred_id])

    top_ids = np.argsort(proba)[::-1][:5]

    top_candidates = []

    for idx in top_ids:
        aa = label_encoder.inverse_transform([idx])[0]
        top_candidates.append({
            "amino_acid": aa,
            "confidence": float(proba[idx])
        })

    return pred_aa, confidence, top_candidates


def fill_known_size_gap_ml(left_context, right_context, gap_size, k=11):
    filled = ""
    residue_confidences = []

    for _ in range(gap_size):
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)

        masked_kmer = prefix + GAP_TOKEN

        pred_aa, conf, _ = predict_masked_kmer(masked_kmer)

        filled += pred_aa
        residue_confidences.append(conf)

    confidence = float(np.mean(residue_confidences)) if residue_confidences else 0.0

    return {
        "prediction": filled,
        "confidence": confidence,
        "mass": peptide_mass(filled)
    }


known_size_example = fill_known_size_gap_ml(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    gap_size=3
)

print("\n==== Known-size Example ====")
print(known_size_example)


# ============================================================
# 13. GENERATE SIMULATED GAP SAMPLES FOR KNOWN-SIZE EVALUATION
# ============================================================

def create_gap_sample(sequence, protein_id, gap_start, gap_len, context_size=15):
    gap_end = gap_start + gap_len
    gap_seq = sequence[gap_start:gap_end]

    left_context = sequence[max(0, gap_start - context_size):gap_start]
    right_context = sequence[gap_end:gap_end + context_size]

    return {
        "protein_id": protein_id,
        "full_sequence": sequence,
        "gap_start": gap_start,
        "gap_end": gap_end,
        "gap_size": gap_len,
        "gap_sequence": gap_seq,
        "gap_mass": peptide_mass(gap_seq),
        "left_context": left_context,
        "right_context": right_context
    }


def generate_gap_dataset(
    sequences,
    protein_prefix,
    samples_per_sequence=2,
    min_gap=1,
    max_gap=8,
    context_size=15
):
    rows = []

    for idx, seq in enumerate(sequences):
        if len(seq) < max_gap + 2 * context_size + 5:
            continue

        for _ in range(samples_per_sequence):
            gap_len = random.randint(min_gap, max_gap)

            min_start = context_size
            max_start = len(seq) - gap_len - context_size

            if max_start <= min_start:
                continue

            gap_start = random.randint(min_start, max_start)

            rows.append(
                create_gap_sample(
                    sequence=seq,
                    protein_id=f"{protein_prefix}_{idx}",
                    gap_start=gap_start,
                    gap_len=gap_len,
                    context_size=context_size
                )
            )

    return pd.DataFrame(rows)


gap_df = pd.concat([
    generate_gap_dataset(mab_train_seqs, "MAB", SAMPLES_PER_SEQUENCE),
    generate_gap_dataset(p5a_train_seqs, "P5A", SAMPLES_PER_SEQUENCE),
    generate_gap_dataset(cah2_train_seqs, "CAH2", SAMPLES_PER_SEQUENCE)
], ignore_index=True)

print("\n==== Known-size Gap Evaluation Samples ====")
print("Gap evaluation samples:", gap_df.shape)
display(gap_df.head())


def residue_accuracy(pred, truth):
    if len(pred) == 0 or len(truth) == 0:
        return 0.0

    n = min(len(pred), len(truth))
    matches = sum(pred[i] == truth[i] for i in range(n))
    return matches / max(len(pred), len(truth))


def evaluate_known_size_gap_filling(df, max_rows=200):
    rows = []

    eval_df = df.head(max_rows).copy()

    for _, row in eval_df.iterrows():
        result = fill_known_size_gap_ml(
            left_context=row["left_context"],
            right_context=row["right_context"],
            gap_size=int(row["gap_size"])
        )

        pred = result["prediction"]
        truth = row["gap_sequence"]

        rows.append({
            "protein_id": row["protein_id"],
            "truth": truth,
            "prediction": pred,
            "gap_size": row["gap_size"],
            "truth_mass": row["gap_mass"],
            "pred_mass": result["mass"],
            "confidence": result["confidence"],
            "exact_match": pred == truth,
            "residue_accuracy": residue_accuracy(pred, truth)
        })

    return pd.DataFrame(rows)


known_size_eval_df = evaluate_known_size_gap_filling(
    gap_df,
    max_rows=200
)

print("\n==== Known-size Evaluation ====")
display(known_size_eval_df.head(20))
print("Known-size exact match accuracy:", known_size_eval_df["exact_match"].mean())
print("Known-size mean residue accuracy:", known_size_eval_df["residue_accuracy"].mean())


# ============================================================
# 14. KNOWN-MASS PROBABILISTIC CANDIDATE GENERATION
# ============================================================

def generate_mass_candidates_from_homologs(
    homolog_sequences,
    target_mass,
    tolerance=0,
    min_len=None,
    max_len=None
):
    if min_len is None:
        min_len = max(
            1,
            math.floor((target_mass - tolerance) / max(AA_MASS.values()))
        )

    if max_len is None:
        max_len = math.ceil(
            (target_mass + tolerance) / min(AA_MASS.values())
        )

    candidate_counter = Counter()

    for seq in homolog_sequences:
        seq = str(seq).strip().upper()
        n = len(seq)

        for length in range(min_len, max_len + 1):
            if n < length:
                continue

            for i in range(n - length + 1):
                kmer = seq[i:i+length]

                if not is_valid_protein_sequence(kmer):
                    continue

                mass = peptide_mass(kmer)

                if abs(mass - target_mass) <= tolerance:
                    candidate_counter[kmer] += 1

    return candidate_counter


def context_support_score(
    candidate,
    left_context,
    right_context,
    homolog_sequences,
    flank=5
):
    left_context = str(left_context).upper()
    right_context = str(right_context).upper()

    left = left_context[-flank:] if left_context else ""
    right = right_context[:flank] if right_context else ""

    pattern_full = left + candidate + right
    pattern_left = left + candidate
    pattern_right = candidate + right

    full_count = 0
    left_count = 0
    right_count = 0
    candidate_count = 0

    for seq in homolog_sequences:
        seq = str(seq).upper()

        full_count += seq.count(pattern_full)
        left_count += seq.count(pattern_left)
        right_count += seq.count(pattern_right)
        candidate_count += seq.count(candidate)

    score = (
        5.0 * full_count +
        2.0 * left_count +
        2.0 * right_count +
        1.0 * candidate_count
    )

    return float(score)


def probabilistic_rank_candidates(
    target_mass,
    left_context,
    right_context,
    homolog_sequences,
    tolerance=0,
    top_k=50
):
    candidates = generate_mass_candidates_from_homologs(
        homolog_sequences=homolog_sequences,
        target_mass=target_mass,
        tolerance=tolerance
    )

    rows = []

    for candidate, frequency in candidates.items():
        mass = peptide_mass(candidate)
        mass_error = abs(mass - target_mass)

        context_score = context_support_score(
            candidate=candidate,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            flank=5
        )

        probabilistic_score = (
            frequency +
            context_score -
            mass_error
        )

        rows.append({
            "candidate": candidate,
            "candidate_length": len(candidate),
            "candidate_mass": mass,
            "target_mass": target_mass,
            "mass_error": mass_error,
            "homolog_frequency": frequency,
            "context_score": context_score,
            "probabilistic_score": probabilistic_score,
            "mass_valid": mass_error <= tolerance
        })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        return df

    return df.sort_values(
        "probabilistic_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)


print("\nKnown-mass probabilistic ranking functions loaded.")


# ============================================================
# 15. HYBRID RERANKING USING ML ENSEMBLE
# ============================================================

def ml_score_candidate(left_context, right_context, candidate, k=11):
    if len(candidate) == 0:
        return 0.0

    scores = []
    filled = ""

    for aa in candidate:
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)

        masked_kmer = prefix + GAP_TOKEN

        x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)

        proba = weighted_ensemble_predict_proba(
            top_base8_wrapped_models,
            x
        )[0]

        if aa in label_encoder.classes_:
            aa_id = label_encoder.transform([aa])[0]
            scores.append(float(proba[aa_id]))
        else:
            scores.append(0.0)

        filled += aa

    return float(np.mean(scores)) if scores else 0.0


def hybrid_rerank_candidates(
    candidate_df,
    left_context,
    right_context,
    target_mass,
    mass_tolerance=0
):
    if candidate_df is None or len(candidate_df) == 0:
        return pd.DataFrame()

    rows = []

    for _, row in candidate_df.iterrows():
        candidate = row["candidate"]

        ml_score = ml_score_candidate(
            left_context=left_context,
            right_context=right_context,
            candidate=candidate,
            k=11
        )

        candidate_mass = peptide_mass(candidate)
        mass_error = abs(candidate_mass - target_mass)

        mass_valid_score = 1.0 if mass_error <= mass_tolerance else 0.0
        homolog_frequency_score = math.log1p(row["homolog_frequency"])
        context_score_norm = math.log1p(row["context_score"])

        hybrid_score = (
            3.0 * mass_valid_score +
            2.0 * homolog_frequency_score +
            2.0 * context_score_norm +
            2.0 * ml_score -
            2.0 * mass_error
        )

        out = row.to_dict()
        out.update({
            "ml_ensemble_score": ml_score,
            "mass_valid_score": mass_valid_score,
            "hybrid_score": hybrid_score
        })

        rows.append(out)

    out_df = pd.DataFrame(rows)

    return out_df.sort_values(
        "hybrid_score",
        ascending=False
    ).reset_index(drop=True)


# ============================================================
# 16. CAH2 KNOWN-MASS TEST CASES
# ============================================================

cah2_test_cases = [
    {
        "gap_name": "CAH2_gap_1",
        "mass": 420,
        "left": "MSHHWGYGKHN",
        "right": "WHKDFPIAKGER",
        "truth": "GPEH"
    },
    {
        "gap_name": "CAH2_gap_2",
        "mass": 750,
        "left": "DPSLKPL",
        "right": "TSLRILNNGHA",
        "truth": "SVSYDQA"
    },
    {
        "gap_name": "CAH2_gap_3",
        "mass": 622,
        "left": "SQDK",
        "right": "LDGTYRLIQF",
        "truth": "AVLKGGP"
    },
    {
        "gap_name": "CAH2_gap_4",
        "mass": 1318,
        "left": "VHWNT",
        "right": "DGLAVLGIFL",
        "truth": "KYGDFGKAVQQP"
    },
    {
        "gap_name": "CAH2_gap_5",
        "mass": 751,
        "left": "DYWTYPGSL",
        "right": "CVTWIVLKEP",
        "truth": "TTPPLLE"
    },
    {
        "gap_name": "CAH2_gap_6",
        "mass": 544,
        "left": "VSSEQV",
        "right": "KLNFNGEGEP",
        "truth": "LKFR"
    },
    {
        "gap_name": "CAH2_gap_7",
        "mass": 561,
        "left": "PLKNRQI",
        "right": "",
        "truth": "KASFK"
    }
]


# ============================================================
# 17. TEST PROBABILISTIC FUNCTION
# ============================================================

print("\n==== Test Known-mass Candidate Generation ====")

test_candidates = probabilistic_rank_candidates(
    target_mass=420,
    left_context="MSHHWGYGKHN",
    right_context="WHKDFPIAKGER",
    homolog_sequences=cah2_train_seqs,
    tolerance=0,
    top_k=10
)

display(test_candidates)


# ============================================================
# 18. EVALUATE KNOWN-MASS HYBRID MODEL
# ============================================================

known_mass_results = []

for case in cah2_test_cases:
    print("\nRunning:", case["gap_name"], "mass:", case["mass"])

    prob_candidates = probabilistic_rank_candidates(
        target_mass=case["mass"],
        left_context=case["left"],
        right_context=case["right"],
        homolog_sequences=cah2_train_seqs,
        tolerance=0,
        top_k=50
    )

    hybrid_candidates = hybrid_rerank_candidates(
        candidate_df=prob_candidates,
        left_context=case["left"],
        right_context=case["right"],
        target_mass=case["mass"],
        mass_tolerance=0
    )

    if len(hybrid_candidates) == 0:
        pred = ""
        pred_mass = None
        hybrid_score = None
        rank_of_truth = None
        top_5 = []
    else:
        pred = hybrid_candidates.iloc[0]["candidate"]
        pred_mass = hybrid_candidates.iloc[0]["candidate_mass"]
        hybrid_score = hybrid_candidates.iloc[0]["hybrid_score"]
        top_5 = hybrid_candidates["candidate"].head(5).tolist()

        truth_positions = hybrid_candidates.index[
            hybrid_candidates["candidate"] == case["truth"]
        ].tolist()

        rank_of_truth = truth_positions[0] + 1 if truth_positions else None

    known_mass_results.append({
        "gap_name": case["gap_name"],
        "target_mass": case["mass"],
        "truth": case["truth"],
        "prediction": pred,
        "pred_mass": pred_mass,
        "hybrid_score": hybrid_score,
        "exact_match": pred == case["truth"],
        "rank_of_truth": rank_of_truth,
        "top_5_candidates": top_5
    })

    display(hybrid_candidates.head(10))

known_mass_results_df = pd.DataFrame(known_mass_results)

print("\n==== Known-mass Evaluation ====")
display(known_mass_results_df)

print("Known-mass exact match accuracy:", known_mass_results_df["exact_match"].mean())

known_mass_top5 = known_mass_results_df["rank_of_truth"].apply(
    lambda x: x is not None and x <= 5
).mean()

print("Known-mass top-5 recovery:", known_mass_top5)


# ============================================================
# 19. FINAL INFERENCE FUNCTION
# ============================================================

def final_predict_gap(
    left_context,
    right_context,
    known_gap_size=None,
    known_gap_mass=None,
    homolog_sequences=None,
    mass_tolerance=0,
    top_k=10
):
    if homolog_sequences is None:
        homolog_sequences = all_homologs

    if known_gap_mass is not None:
        prob_candidates = probabilistic_rank_candidates(
            target_mass=known_gap_mass,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            tolerance=mass_tolerance,
            top_k=100
        )

        if known_gap_size is not None and len(prob_candidates) > 0:
            prob_candidates = prob_candidates[
                prob_candidates["candidate_length"] == known_gap_size
            ].reset_index(drop=True)

        hybrid_candidates = hybrid_rerank_candidates(
            candidate_df=prob_candidates,
            left_context=left_context,
            right_context=right_context,
            target_mass=known_gap_mass,
            mass_tolerance=mass_tolerance
        )

        if len(hybrid_candidates) == 0:
            return {
                "mode": "known_mass_hybrid",
                "prediction": "",
                "confidence": 0.0,
                "top_candidates": []
            }

        best = hybrid_candidates.iloc[0]

        return {
            "mode": "known_mass_hybrid",
            "prediction": best["candidate"],
            "confidence": float(best["hybrid_score"]),
            "mass_valid": bool(best["mass_valid"]),
            "pred_mass": int(best["candidate_mass"]),
            "target_mass": int(known_gap_mass),
            "top_candidates": hybrid_candidates.head(top_k).to_dict("records")
        }

    if known_gap_size is not None:
        result = fill_known_size_gap_ml(
            left_context=left_context,
            right_context=right_context,
            gap_size=known_gap_size
        )

        return {
            "mode": "known_size_ml_ensemble",
            "prediction": result["prediction"],
            "confidence": result["confidence"],
            "pred_mass": result["mass"]
        }

    raise ValueError("Provide known_gap_size or known_gap_mass.")


print("\n==== Final Inference Tests ====")

known_size_result = final_predict_gap(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    known_gap_size=3
)

print("Known-size result:")
print(known_size_result)

known_mass_result = final_predict_gap(
    left_context="MSHHWGYGKHN",
    right_context="WHKDFPIAKGER",
    known_gap_mass=420,
    homolog_sequences=cah2_train_seqs,
    mass_tolerance=0,
    top_k=5
)

print("\nKnown-mass result:")
print("Prediction:", known_mass_result["prediction"])
print("Mass valid:", known_mass_result["mass_valid"])
display(pd.DataFrame(known_mass_result["top_candidates"]).head())


# ============================================================
# 20. SAVE OUTPUT FILES
# ============================================================

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

base8_all_results.to_csv(
    f"{OUTPUT_DIR}/base8_all_model_results.csv",
    index=False
)

top_base8_models_df.to_csv(
    f"{OUTPUT_DIR}/base8_selected_top_models.csv",
    index=False
)

known_size_eval_df.to_csv(
    f"{OUTPUT_DIR}/known_size_gap_evaluation.csv",
    index=False
)

known_mass_results_df.to_csv(
    f"{OUTPUT_DIR}/known_mass_gap_evaluation.csv",
    index=False
)

summary_df = pd.DataFrame([
    {
        "system": "Best single base 8 model",
        "metric": "validation_accuracy",
        "score": base8_all_results.iloc[0]["validation_accuracy"]
    },
    {
        "system": f"Weighted ensemble top {TOP_N_MODELS}",
        "metric": "validation_accuracy",
        "score": ensemble_val_acc
    },
    {
        "system": "Known-size iterative ML ensemble",
        "metric": "exact_match_accuracy",
        "score": known_size_eval_df["exact_match"].mean()
    },
    {
        "system": "Known-size iterative ML ensemble",
        "metric": "mean_residue_accuracy",
        "score": known_size_eval_df["residue_accuracy"].mean()
    },
    {
        "system": "Known-mass hybrid model",
        "metric": "exact_match_accuracy",
        "score": known_mass_results_df["exact_match"].mean()
    },
    {
        "system": "Known-mass hybrid model",
        "metric": "top_5_recovery",
        "score": known_mass_top5
    }
])

summary_df.to_csv(
    f"{OUTPUT_DIR}/final_summary.csv",
    index=False
)

print("\n==== Final Summary ====")
display(summary_df)

print("\n==== Saved Files ====")
for file in os.listdir(OUTPUT_DIR):
    if file.endswith(".csv"):
        print(os.path.join(OUTPUT_DIR, file))

==== Dataset Paths ====
DE_NOVO_CAH2 => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/de_novo_sequence_CAH2.txt | exists: True
MAB_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_target_sequence.txt | exists: True
MAB_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_training_sequence.txt | exists: True
P5A_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/p5A_training_sequence.txt | exists: True
CAH2_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/target_sequence_CAH2.txt | exists: True
CAH2_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/training_sequences_CAH2.txt | exists: True

==== Amino Acid Checks ====
Mass GPEH: 420
Mass SVSYDQA: 750

==== Loaded Sequences ====
Mab target length: 214
Mab training sequences: 992
P5A training sequences: 1000
CAH2 target length: 260
CAH2 training sequences: 1000
All homolog

,model_name,base_model,feature_type,train_accuracy,validation_accuracy,train_time_sec
0,raw__ExtraTrees,ExtraTrees,raw,0.979000,0.960833,0.979558
1,raw__RandomForest,RandomForest,raw,0.979000,0.960833,0.990532
2,raw__HistGradientBoosting,HistGradientBoosting,raw,0.974458,0.959000,13.270295
3,raw__MLP,MLP,raw,0.974542,0.938167,33.083281
4,raw__DecisionTree,DecisionTree,raw,0.963250,0.937667,0.072333
5,svd__ExtraTrees,ExtraTrees,svd,0.979000,0.936667,1.225911
6,svd__RandomForest,RandomForest,svd,0.979000,0.931000,1.933322
7,svd__HistGradientBoosting,HistGradientBoosting,svd,0.971125,0.930333,15.533311
8,raw__kNN,kNN,raw,0.943458,0.929500,0.775607
9,svd__kNN,kNN,svd,0.922875,0.903167,0.457765



==== Selected Top Models ====


,model_name,base_model,feature_type,train_accuracy,validation_accuracy,train_time_sec
0,raw__ExtraTrees,ExtraTrees,raw,0.979000,0.960833,0.979558
1,raw__RandomForest,RandomForest,raw,0.979000,0.960833,0.990532
2,raw__HistGradientBoosting,HistGradientBoosting,raw,0.974458,0.959000,13.270295
3,raw__MLP,MLP,raw,0.974542,0.938167,33.083281
4,raw__DecisionTree,DecisionTree,raw,0.963250,0.937667,0.072333



==== Models Used in Ensemble ====
raw__ExtraTrees | acc: 0.9608333333333333
raw__RandomForest | acc: 0.9608333333333333
raw__HistGradientBoosting | acc: 0.959
raw__MLP | acc: 0.9381666666666667
raw__DecisionTree | acc: 0.9376666666666666

==== Ensemble Validation ====
Best single model accuracy: 0.9608333333333333
Weighted ensemble accuracy: 0.9585

==== Ensemble Top-k Accuracy ====
Top-1 accuracy: 0.9585
Top-2 accuracy: 0.9768
Top-3 accuracy: 0.9825
Top-5 accuracy: 0.9872

==== Known-size Example ====
{'prediction': 'SAS', 'confidence': 0.9968581758076426, 'mass': 245}

==== Known-size Gap Evaluation Samples ====
Gap evaluation samples: (5984, 9)


...

Known-mass exact match accuracy: 0.7142857142857143
Known-mass top-5 recovery: 1.0

==== Final Inference Tests ====
Known-size result:
{'mode': 'known_size_ml_ensemble', 'prediction': 'SAS', 'confidence': 0.9968581758076426, 'pred_mass': 245}

Known-mass result:
Prediction: GPEH
Mass valid: True


,candidate,candidate_length,candidate_mass,target_mass,mass_error,homolog_frequency,context_score,probabilistic_score,mass_valid,ml_ensemble_score,mass_valid_score,hybrid_score
0,GPEH,4,420,420,0,366,2559.0,2925.0,True,0.075470,1.0,30.657189
1,PPLL,4,420,420,0,909,909.0,1818.0,True,0.065206,1.0,30.384191
2,TYR,3,420,420,0,540,540.0,1080.0,True,0.061986,1.0,28.297648
3,GLCF,4,420,420,0,391,391.0,782.0,True,0.068302,1.0,27.021652
4,SADF,4,420,420,0,333,333.0,666.0,True,0.017679,1.0,26.279922



==== Final Summary ====


,system,metric,score
0,Best single base 8 model,validation_accuracy,0.960833
1,Weighted ensemble top 5,validation_accuracy,0.958500
2,Known-size iterative ML ensemble,exact_match_accuracy,0.860000
3,Known-size iterative ML ensemble,mean_residue_accuracy,0.943351
4,Known-mass hybrid model,exact_match_accuracy,0.714286
5,Known-mass hybrid model,top_5_recovery,1.000000



==== Saved Files ====
/kaggle/working/base8_selected_top_models.csv
/kaggle/working/final_summary.csv
/kaggle/working/known_mass_gap_evaluation.csv
/kaggle/working/base8_all_model_results.csv
/kaggle/working/known_size_gap_evaluation.csv


In [ ]:
# ============================================================
# ADVANCED PERFORMANCE UPGRADE CELL
# Run this AFTER your previous full one-cell code.
#
# Improvements:
# 1. Uses best single ML model if ensemble is weaker.
# 2. Adds exact left/right/full-context anchoring.
# 3. Improves known-mass reranking.
# 4. Improves known-size gap filling using candidate reranking.
# 5. Saves upgraded evaluation results.
# ============================================================

import os
import math
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.metrics import accuracy_score


# ============================================================
# 1. Choose strongest ML predictor automatically
# ============================================================

best_single_model_name = base8_all_results.iloc[0]["model_name"]
best_single_feature_type = base8_all_results.iloc[0]["feature_type"]
best_single_acc = base8_all_results.iloc[0]["validation_accuracy"]

best_single_wrapper = Base8ModelWrapper(
    model_name=best_single_model_name,
    model=base8_all_fitted[best_single_model_name],
    feature_type=best_single_feature_type,
    validation_accuracy=best_single_acc
)

if ensemble_val_acc >= best_single_acc:
    ACTIVE_ML_MODE = "ensemble"
else:
    ACTIVE_ML_MODE = "best_single"

print("Active ML mode:", ACTIVE_ML_MODE)
print("Best single:", best_single_model_name, best_single_acc)
print("Ensemble:", ensemble_val_acc)


def active_predict_proba(X):
    if ACTIVE_ML_MODE == "ensemble":
        return weighted_ensemble_predict_proba(top_base8_wrapped_models, X)
    return best_single_wrapper.predict_proba(X)


def active_predict(X):
    proba = active_predict_proba(X)
    return np.argmax(proba, axis=1)


# ============================================================
# 2. Faster homolog text and k-mer index
# ============================================================

ALL_HOMOLOG_TEXT = "|".join(all_homologs)
CAH2_HOMOLOG_TEXT = "|".join(cah2_train_seqs)

MAX_INDEX_K = 15


def build_kmer_index(sequences, max_k=15):
    index = {k: Counter() for k in range(1, max_k + 1)}

    for seq in sequences:
        seq = str(seq).strip().upper()

        for k in range(1, max_k + 1):
            if len(seq) < k:
                continue

            for i in range(len(seq) - k + 1):
                kmer = seq[i:i+k]
                if is_valid_protein_sequence(kmer):
                    index[k][kmer] += 1

    return index


print("Building k-mer index...")
ALL_KMER_INDEX = build_kmer_index(all_homologs, max_k=MAX_INDEX_K)
CAH2_KMER_INDEX = build_kmer_index(cah2_train_seqs, max_k=MAX_INDEX_K)
print("K-mer index ready.")


# ============================================================
# 3. Advanced context-count scoring
# ============================================================

def advanced_context_counts(candidate, left_context, right_context, homolog_text, flank=5):
    left_context = str(left_context).upper()
    right_context = str(right_context).upper()
    candidate = str(candidate).upper()

    left = left_context[-flank:] if left_context else ""
    right = right_context[:flank] if right_context else ""

    full_pattern = left + candidate + right
    left_pattern = left + candidate
    right_pattern = candidate + right

    full_count = homolog_text.count(full_pattern) if left and right else 0
    left_count = homolog_text.count(left_pattern) if left else 0
    right_count = homolog_text.count(right_pattern) if right else 0
    candidate_count = homolog_text.count(candidate)

    return {
        "full_count": full_count,
        "left_count": left_count,
        "right_count": right_count,
        "candidate_count": candidate_count
    }


def ml_score_candidate_advanced(left_context, right_context, candidate, k=11):
    if len(candidate) == 0:
        return 0.0

    scores = []
    filled = ""

    for aa in candidate:
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)
        masked_kmer = prefix + GAP_TOKEN

        x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)
        proba = active_predict_proba(x)[0]

        if aa in label_encoder.classes_:
            aa_id = label_encoder.transform([aa])[0]
            scores.append(float(proba[aa_id]))
        else:
            scores.append(0.0)

        filled += aa

    return float(np.mean(scores)) if scores else 0.0


# ============================================================
# 4. Advanced known-mass candidate generation
# ============================================================

def generate_mass_candidates_from_index(
    kmer_index,
    target_mass,
    tolerance=0
):
    rows = []

    min_len = max(
        1,
        math.floor((target_mass - tolerance) / max(AA_MASS.values()))
    )

    max_len = min(
        max(kmer_index.keys()),
        math.ceil((target_mass + tolerance) / min(AA_MASS.values()))
    )

    for length in range(min_len, max_len + 1):
        for candidate, freq in kmer_index.get(length, {}).items():
            mass = peptide_mass(candidate)

            if abs(mass - target_mass) <= tolerance:
                rows.append({
                    "candidate": candidate,
                    "candidate_length": len(candidate),
                    "candidate_mass": mass,
                    "target_mass": target_mass,
                    "mass_error": abs(mass - target_mass),
                    "homolog_frequency": freq,
                    "mass_valid": abs(mass - target_mass) <= tolerance
                })

    return pd.DataFrame(rows)


def advanced_known_mass_ranker(
    target_mass,
    left_context,
    right_context,
    homolog_text,
    kmer_index,
    tolerance=0,
    top_k=20,
    use_ml=True
):
    candidate_df = generate_mass_candidates_from_index(
        kmer_index=kmer_index,
        target_mass=target_mass,
        tolerance=tolerance
    )

    if len(candidate_df) == 0:
        return candidate_df

    rows = []

    for _, row in candidate_df.iterrows():
        candidate = row["candidate"]

        counts = advanced_context_counts(
            candidate=candidate,
            left_context=left_context,
            right_context=right_context,
            homolog_text=homolog_text,
            flank=5
        )

        ml_score = ml_score_candidate_advanced(
            left_context=left_context,
            right_context=right_context,
            candidate=candidate,
            k=11
        ) if use_ml else 0.0

        mass_error = row["mass_error"]
        mass_valid_score = 1.0 if mass_error <= tolerance else 0.0

        # Stronger context-aware score:
        # full_count is the strongest evidence,
        # left/right anchoring is next,
        # frequency is useful but should not dominate.
        advanced_score = (
            100.0 * counts["full_count"] +
            30.0 * counts["left_count"] +
            30.0 * counts["right_count"] +
            3.0 * math.log1p(counts["candidate_count"]) +
            1.0 * math.log1p(row["homolog_frequency"]) +
            5.0 * ml_score +
            10.0 * mass_valid_score -
            10.0 * mass_error
        )

        out = row.to_dict()
        out.update(counts)
        out.update({
            "ml_score": ml_score,
            "advanced_score": advanced_score
        })

        rows.append(out)

    ranked = pd.DataFrame(rows).sort_values(
        "advanced_score",
        ascending=False
    ).reset_index(drop=True)

    return ranked.head(top_k)


# ============================================================
# 5. Advanced known-size candidate reranker
# ============================================================

def anchored_candidates_for_known_size(
    left_context,
    right_context,
    gap_size,
    homolog_sequences,
    flank=5
):
    left = str(left_context).upper()[-flank:] if left_context else ""
    right = str(right_context).upper()[:flank] if right_context else ""

    anchored = Counter()

    if not left and not right:
        return anchored

    for seq in homolog_sequences:
        seq = str(seq).upper()

        if left and right:
            start = 0
            while True:
                pos = seq.find(left, start)
                if pos == -1:
                    break

                cand_start = pos + len(left)
                cand_end = cand_start + gap_size
                right_start = cand_end
                right_end = right_start + len(right)

                if right_end <= len(seq):
                    cand = seq[cand_start:cand_end]
                    observed_right = seq[right_start:right_end]

                    if observed_right == right and is_valid_protein_sequence(cand):
                        anchored[cand] += 1

                start = pos + 1

        elif left:
            start = 0
            while True:
                pos = seq.find(left, start)
                if pos == -1:
                    break

                cand_start = pos + len(left)
                cand_end = cand_start + gap_size

                if cand_end <= len(seq):
                    cand = seq[cand_start:cand_end]
                    if is_valid_protein_sequence(cand):
                        anchored[cand] += 1

                start = pos + 1

        elif right:
            start = 0
            while True:
                pos = seq.find(right, start)
                if pos == -1:
                    break

                cand_end = pos
                cand_start = cand_end - gap_size

                if cand_start >= 0:
                    cand = seq[cand_start:cand_end]
                    if is_valid_protein_sequence(cand):
                        anchored[cand] += 1

                start = pos + 1

    return anchored


def iterative_ml_candidate(left_context, right_context, gap_size):
    return fill_known_size_gap_ml(
        left_context=left_context,
        right_context=right_context,
        gap_size=gap_size
    )["prediction"]


def advanced_known_size_ranker(
    left_context,
    right_context,
    gap_size,
    homolog_sequences,
    homolog_text,
    kmer_index,
    top_k=10
):
    candidates = Counter()

    # 1. Strong anchored candidates from local homolog context
    anchored = anchored_candidates_for_known_size(
        left_context=left_context,
        right_context=right_context,
        gap_size=gap_size,
        homolog_sequences=homolog_sequences,
        flank=5
    )
    candidates.update(anchored)

    # 2. Add frequent candidates of same length
    for cand, freq in kmer_index.get(gap_size, {}).most_common(300):
        candidates[cand] += freq

    # 3. Add ML iterative prediction
    ml_iter = iterative_ml_candidate(
        left_context=left_context,
        right_context=right_context,
        gap_size=gap_size
    )
    if ml_iter and is_valid_protein_sequence(ml_iter):
        candidates[ml_iter] += 1

    rows = []

    for cand, freq in candidates.items():
        if len(cand) != gap_size or not is_valid_protein_sequence(cand):
            continue

        counts = advanced_context_counts(
            candidate=cand,
            left_context=left_context,
            right_context=right_context,
            homolog_text=homolog_text,
            flank=5
        )

        ml_score = ml_score_candidate_advanced(
            left_context=left_context,
            right_context=right_context,
            candidate=cand,
            k=11
        )

        score = (
            120.0 * counts["full_count"] +
            35.0 * counts["left_count"] +
            35.0 * counts["right_count"] +
            2.0 * math.log1p(counts["candidate_count"]) +
            1.0 * math.log1p(freq) +
            10.0 * ml_score
        )

        rows.append({
            "candidate": cand,
            "frequency": freq,
            "candidate_mass": peptide_mass(cand),
            "ml_score": ml_score,
            "advanced_score": score,
            **counts
        })

    ranked = pd.DataFrame(rows)

    if len(ranked) == 0:
        fallback = iterative_ml_candidate(left_context, right_context, gap_size)
        return pd.DataFrame([{
            "candidate": fallback,
            "frequency": 0,
            "candidate_mass": peptide_mass(fallback),
            "ml_score": 0.0,
            "advanced_score": 0.0,
            "full_count": 0,
            "left_count": 0,
            "right_count": 0,
            "candidate_count": 0
        }])

    return ranked.sort_values(
        "advanced_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)


def fill_known_size_gap_advanced(left_context, right_context, gap_size):
    ranked = advanced_known_size_ranker(
        left_context=left_context,
        right_context=right_context,
        gap_size=gap_size,
        homolog_sequences=all_homologs,
        homolog_text=ALL_HOMOLOG_TEXT,
        kmer_index=ALL_KMER_INDEX,
        top_k=10
    )

    best = ranked.iloc[0]

    return {
        "prediction": best["candidate"],
        "confidence": best["advanced_score"],
        "mass": int(best["candidate_mass"]),
        "top_candidates": ranked
    }


# ============================================================
# 6. Re-evaluate known-size with advanced reranker
# ============================================================

def evaluate_known_size_gap_filling_advanced(df, max_rows=200):
    rows = []

    eval_df = df.head(max_rows).copy()

    for _, row in eval_df.iterrows():
        result = fill_known_size_gap_advanced(
            left_context=row["left_context"],
            right_context=row["right_context"],
            gap_size=int(row["gap_size"])
        )

        pred = result["prediction"]
        truth = row["gap_sequence"]

        rows.append({
            "protein_id": row["protein_id"],
            "truth": truth,
            "prediction": pred,
            "gap_size": row["gap_size"],
            "truth_mass": row["gap_mass"],
            "pred_mass": result["mass"],
            "confidence": result["confidence"],
            "exact_match": pred == truth,
            "residue_accuracy": residue_accuracy(pred, truth)
        })

    return pd.DataFrame(rows)


print("\nRunning upgraded known-size evaluation...")
known_size_advanced_eval_df = evaluate_known_size_gap_filling_advanced(
    gap_df,
    max_rows=200
)

display(known_size_advanced_eval_df.head(20))

print("Old known-size exact match:", known_size_eval_df["exact_match"].mean())
print("New known-size exact match:", known_size_advanced_eval_df["exact_match"].mean())

print("Old known-size residue accuracy:", known_size_eval_df["residue_accuracy"].mean())
print("New known-size residue accuracy:", known_size_advanced_eval_df["residue_accuracy"].mean())


# ============================================================
# 7. Re-evaluate known-mass with advanced reranker
# ============================================================

advanced_known_mass_results = []

for case in cah2_test_cases:
    print("\nAdvanced known-mass running:", case["gap_name"], "mass:", case["mass"])

    ranked = advanced_known_mass_ranker(
        target_mass=case["mass"],
        left_context=case["left"],
        right_context=case["right"],
        homolog_text=CAH2_HOMOLOG_TEXT,
        kmer_index=CAH2_KMER_INDEX,
        tolerance=0,
        top_k=20,
        use_ml=True
    )

    if len(ranked) == 0:
        pred = ""
        pred_mass = None
        score = None
        rank_of_truth = None
        top_5 = []
    else:
        pred = ranked.iloc[0]["candidate"]
        pred_mass = ranked.iloc[0]["candidate_mass"]
        score = ranked.iloc[0]["advanced_score"]
        top_5 = ranked["candidate"].head(5).tolist()

        truth_positions = ranked.index[ranked["candidate"] == case["truth"]].tolist()
        rank_of_truth = truth_positions[0] + 1 if truth_positions else None

    advanced_known_mass_results.append({
        "gap_name": case["gap_name"],
        "target_mass": case["mass"],
        "truth": case["truth"],
        "prediction": pred,
        "pred_mass": pred_mass,
        "advanced_score": score,
        "exact_match": pred == case["truth"],
        "rank_of_truth": rank_of_truth,
        "top_5_candidates": top_5
    })

    display(ranked.head(10))


known_mass_advanced_results_df = pd.DataFrame(advanced_known_mass_results)

display(known_mass_advanced_results_df)

old_known_mass_acc = known_mass_results_df["exact_match"].mean()
new_known_mass_acc = known_mass_advanced_results_df["exact_match"].mean()

old_known_mass_top5 = known_mass_top5
new_known_mass_top5 = known_mass_advanced_results_df["rank_of_truth"].apply(
    lambda x: x is not None and x <= 5
).mean()

print("Old known-mass exact match:", old_known_mass_acc)
print("New known-mass exact match:", new_known_mass_acc)

print("Old known-mass top-5:", old_known_mass_top5)
print("New known-mass top-5:", new_known_mass_top5)


# ============================================================
# 8. Upgraded final inference function
# ============================================================

def final_predict_gap_best(
    left_context,
    right_context,
    known_gap_size=None,
    known_gap_mass=None,
    homolog_sequences=None,
    homolog_text=None,
    kmer_index=None,
    mass_tolerance=0,
    top_k=10
):
    if homolog_sequences is None:
        homolog_sequences = all_homologs

    if homolog_text is None:
        homolog_text = ALL_HOMOLOG_TEXT

    if kmer_index is None:
        kmer_index = ALL_KMER_INDEX

    if known_gap_mass is not None:
        ranked = advanced_known_mass_ranker(
            target_mass=known_gap_mass,
            left_context=left_context,
            right_context=right_context,
            homolog_text=homolog_text,
            kmer_index=kmer_index,
            tolerance=mass_tolerance,
            top_k=top_k,
            use_ml=True
        )

        if known_gap_size is not None and len(ranked) > 0:
            ranked = ranked[ranked["candidate_length"] == known_gap_size].reset_index(drop=True)

        if len(ranked) == 0:
            return {
                "mode": "advanced_known_mass",
                "prediction": "",
                "confidence": 0.0,
                "top_candidates": []
            }

        best = ranked.iloc[0]

        return {
            "mode": "advanced_known_mass",
            "prediction": best["candidate"],
            "confidence": float(best["advanced_score"]),
            "mass_valid": bool(best["mass_valid"]),
            "pred_mass": int(best["candidate_mass"]),
            "target_mass": int(known_gap_mass),
            "top_candidates": ranked.head(top_k).to_dict("records")
        }

    if known_gap_size is not None:
        ranked = advanced_known_size_ranker(
            left_context=left_context,
            right_context=right_context,
            gap_size=known_gap_size,
            homolog_sequences=homolog_sequences,
            homolog_text=homolog_text,
            kmer_index=kmer_index,
            top_k=top_k
        )

        best = ranked.iloc[0]

        return {
            "mode": "advanced_known_size",
            "prediction": best["candidate"],
            "confidence": float(best["advanced_score"]),
            "pred_mass": int(best["candidate_mass"]),
            "top_candidates": ranked.head(top_k).to_dict("records")
        }

    raise ValueError("Provide known_gap_size or known_gap_mass.")


print("\nAdvanced final inference test:")

test_best_mass = final_predict_gap_best(
    left_context="DPSLKPL",
    right_context="TSLRILNNGHA",
    known_gap_mass=750,
    homolog_sequences=cah2_train_seqs,
    homolog_text=CAH2_HOMOLOG_TEXT,
    kmer_index=CAH2_KMER_INDEX,
    mass_tolerance=0,
    top_k=5
)

print(test_best_mass["prediction"])
display(pd.DataFrame(test_best_mass["top_candidates"]))


# ============================================================
# 9. Save upgraded results
# ============================================================

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

known_size_advanced_eval_df.to_csv(
    f"{OUTPUT_DIR}/known_size_advanced_evaluation.csv",
    index=False
)

known_mass_advanced_results_df.to_csv(
    f"{OUTPUT_DIR}/known_mass_advanced_evaluation.csv",
    index=False
)

advanced_summary_df = pd.DataFrame([
    {
        "system": "Original best single ML model",
        "metric": "validation_accuracy",
        "score": best_single_acc
    },
    {
        "system": "Original weighted ensemble",
        "metric": "validation_accuracy",
        "score": ensemble_val_acc
    },
    {
        "system": "Original known-size iterative ML ensemble",
        "metric": "exact_match_accuracy",
        "score": known_size_eval_df["exact_match"].mean()
    },
    {
        "system": "Advanced known-size candidate reranker",
        "metric": "exact_match_accuracy",
        "score": known_size_advanced_eval_df["exact_match"].mean()
    },
    {
        "system": "Original known-size iterative ML ensemble",
        "metric": "mean_residue_accuracy",
        "score": known_size_eval_df["residue_accuracy"].mean()
    },
    {
        "system": "Advanced known-size candidate reranker",
        "metric": "mean_residue_accuracy",
        "score": known_size_advanced_eval_df["residue_accuracy"].mean()
    },
    {
        "system": "Original known-mass hybrid model",
        "metric": "exact_match_accuracy",
        "score": old_known_mass_acc
    },
    {
        "system": "Advanced known-mass context-aware reranker",
        "metric": "exact_match_accuracy",
        "score": new_known_mass_acc
    },
    {
        "system": "Original known-mass hybrid model",
        "metric": "top_5_recovery",
        "score": old_known_mass_top5
    },
    {
        "system": "Advanced known-mass context-aware reranker",
        "metric": "top_5_recovery",
        "score": new_known_mass_top5
    }
])

advanced_summary_df.to_csv(
    f"{OUTPUT_DIR}/advanced_final_summary.csv",
    index=False
)

print("\nADVANCED SUMMARY")
display(advanced_summary_df)

print("\nSaved upgraded files:")
for file in os.listdir(OUTPUT_DIR):
    if file.endswith(".csv") and "advanced" in file:
        print(os.path.join(OUTPUT_DIR, file))

Active ML mode: best_single
Best single: raw__ExtraTrees 0.9608333333333333
Ensemble: 0.9585
Building k-mer index...
K-mer index ready.

Running upgraded known-size evaluation...


In [29]:
import os
import re
import math
import time
import random
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

FAST_MODE = True

if FAST_MODE:
    MAX_ML_SAMPLES = 30000
    SAMPLES_PER_SEQUENCE = 2
    RF_TREES = 100
    ET_TREES = 100
    MLP_MAX_ITER = 150
else:
    MAX_ML_SAMPLES = None
    SAMPLES_PER_SEQUENCE = 5
    RF_TREES = 200
    ET_TREES = 200
    MLP_MAX_ITER = 300

TOP_N_MODELS = 5

DATA_DIR = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets"

DE_NOVO_CAH2 = f"{DATA_DIR}/de_novo_sequence_CAH2.txt"
MAB_TARGET = f"{DATA_DIR}/mab_target_sequence.txt"
MAB_TRAINING = f"{DATA_DIR}/mab_training_sequence.txt"
P5A_TRAINING = f"{DATA_DIR}/p5A_training_sequence.txt"
CAH2_TARGET = f"{DATA_DIR}/target_sequence_CAH2.txt"
CAH2_TRAINING = f"{DATA_DIR}/training_sequences_CAH2.txt"

def find_file(filename, root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

paths = {
    "DE_NOVO_CAH2": ("de_novo_sequence_CAH2.txt", DE_NOVO_CAH2),
    "MAB_TARGET": ("mab_target_sequence.txt", MAB_TARGET),
    "MAB_TRAINING": ("mab_training_sequence.txt", MAB_TRAINING),
    "P5A_TRAINING": ("p5A_training_sequence.txt", P5A_TRAINING),
    "CAH2_TARGET": ("target_sequence_CAH2.txt", CAH2_TARGET),
    "CAH2_TRAINING": ("training_sequences_CAH2.txt", CAH2_TRAINING)
}

fixed_paths = {}

for key, (fname, path) in paths.items():
    if os.path.exists(path):
        fixed_paths[key] = path
    else:
        fixed_paths[key] = find_file(fname)

DE_NOVO_CAH2 = fixed_paths["DE_NOVO_CAH2"]
MAB_TARGET = fixed_paths["MAB_TARGET"]
MAB_TRAINING = fixed_paths["MAB_TRAINING"]
P5A_TRAINING = fixed_paths["P5A_TRAINING"]
CAH2_TARGET = fixed_paths["CAH2_TARGET"]
CAH2_TRAINING = fixed_paths["CAH2_TRAINING"]

for name, path in fixed_paths.items():
    print(name, "=>", path, "| exists:", path is not None and os.path.exists(path))

DE_NOVO_CAH2 => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/de_novo_sequence_CAH2.txt | exists: True
MAB_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_target_sequence.txt | exists: True
MAB_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_training_sequence.txt | exists: True
P5A_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/p5A_training_sequence.txt | exists: True
CAH2_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/target_sequence_CAH2.txt | exists: True
CAH2_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/training_sequences_CAH2.txt | exists: True


In [30]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
GAP_TOKEN = "-"

AA_MASS = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99,
    "T": 101, "C": 103, "L": 113, "I": 113, "N": 114,
    "D": 115, "Q": 128, "K": 128, "E": 129, "M": 131,
    "H": 137, "F": 147, "R": 156, "Y": 163, "W": 186
}

aa_to_int = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa_to_int[GAP_TOKEN] = 0

int_to_aa = {v: k for k, v in aa_to_int.items()}

def peptide_mass(seq):
    return int(sum(AA_MASS.get(aa, 0) for aa in seq))

def is_valid_protein_sequence(seq):
    return all(ch in AMINO_ACIDS for ch in seq)

def encode_sequence(seq):
    return [aa_to_int.get(ch, 0) for ch in seq]

print("Amino acids:", AMINO_ACIDS)
print("Mass GPEH:", peptide_mass("GPEH"))
# print("Mass SVSYDQA:", peptide_mass("SVSYDQA"))

Amino acids: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
Mass GPEH: 420


In [31]:
def read_text(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def clean_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    return seq

def parse_fasta_or_sequences(path):
    text = read_text(path)
    sequences = []
    current = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if current:
                seq = clean_sequence("".join(current))
                if seq:
                    sequences.append(seq)
                current = []
        else:
            current.append(line)

    if current:
        seq = clean_sequence("".join(current))
        if seq:
            sequences.append(seq)

    return sequences

mab_target_seq = parse_fasta_or_sequences(MAB_TARGET)[0]
mab_train_seqs = parse_fasta_or_sequences(MAB_TRAINING)

p5a_train_seqs = parse_fasta_or_sequences(P5A_TRAINING)

cah2_target_seq = parse_fasta_or_sequences(CAH2_TARGET)[0]
cah2_train_seqs = parse_fasta_or_sequences(CAH2_TRAINING)

all_homologs = mab_train_seqs + p5a_train_seqs + cah2_train_seqs

print("Mab target length:", len(mab_target_seq))
print("Mab training sequences:", len(mab_train_seqs))
print("P5A training sequences:", len(p5a_train_seqs))
print("CAH2 target length:", len(cah2_target_seq))
print("CAH2 training sequences:", len(cah2_train_seqs))
print("All homolog sequences:", len(all_homologs))

Mab target length: 214
Mab training sequences: 992
P5A training sequences: 1000
CAH2 target length: 260
CAH2 training sequences: 1000
All homolog sequences: 2992


In [32]:
def generate_masked_kmer_dataset(sequences, k=11, max_samples=None):
    X = []
    y_letters = []

    for seq in sequences:
        if len(seq) < k:
            continue

        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]

            if not is_valid_protein_sequence(kmer):
                continue

            masked_first = GAP_TOKEN + kmer[1:]
            X.append(encode_sequence(masked_first))
            y_letters.append(kmer[0])

            masked_last = kmer[:-1] + GAP_TOKEN
            X.append(encode_sequence(masked_last))
            y_letters.append(kmer[-1])

            if max_samples is not None and len(X) >= max_samples:
                return np.array(X), np.array(y_letters)

    return np.array(X), np.array(y_letters)

X_ml, y_ml_letters = generate_masked_kmer_dataset(
    all_homologs,
    k=11,
    max_samples=MAX_ML_SAMPLES
)

label_encoder = LabelEncoder()
y_ml = label_encoder.fit_transform(y_ml_letters)

X_train_ml, X_val_ml, y_train_ml, y_val_ml = train_test_split(
    X_ml,
    y_ml,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_ml
)

print("X_ml:", X_ml.shape)
print("y_ml:", y_ml.shape)
print("Classes:", list(label_encoder.classes_))
print("Train:", X_train_ml.shape)
print("Validation:", X_val_ml.shape)

X_ml: (30000, 11)
y_ml: (30000,)
Classes: [np.str_('A'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('V'), np.str_('W'), np.str_('Y')]
Train: (24000, 11)
Validation: (6000, 11)


In [33]:
def row_average_features(X):
    return X.mean(axis=1).reshape(-1, 1)

X_train_row = row_average_features(X_train_ml)
X_val_row = row_average_features(X_val_ml)

print("Raw feature shape:", X_train_ml.shape)
print("Row-average shape:", X_train_row.shape)

Raw feature shape: (24000, 11)
Row-average shape: (24000, 1)


In [34]:
def get_base_8_models():
    models = {
        "kNN": KNeighborsClassifier(
            n_neighbors=5
        ),

        "DecisionTree": DecisionTreeClassifier(
            max_depth=20,
            min_samples_split=5,
            random_state=RANDOM_STATE
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=RF_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=ET_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        ),

        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000,
                n_jobs=-1,
                random_state=RANDOM_STATE
            ))
        ]),

        "LinearSVC": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                random_state=RANDOM_STATE
            ))
        ]),

        "MLP": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(150, 75),
                activation="tanh",
                max_iter=MLP_MAX_ITER,
                random_state=RANDOM_STATE
            ))
        ])
    }

    return models

base_8_models = get_base_8_models()

print("Base 8 models:")
for name in base_8_models:
    print("-", name)

Base 8 models:
- kNN
- DecisionTree
- RandomForest
- ExtraTrees
- HistGradientBoosting
- LogisticRegression
- LinearSVC
- MLP


In [35]:
def safe_predict_proba(model, X, num_classes):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)

        if scores.ndim == 1:
            scores = np.vstack([-scores, scores]).T

        scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    preds = model.predict(X)
    proba = np.zeros((len(preds), num_classes))

    for i, p in enumerate(preds):
        proba[i, int(p)] = 1.0

    return proba

def train_and_evaluate_models(models, X_train, X_val, y_train, y_val, feature_type="raw"):
    results = []
    fitted_models = {}

    for name, model in models.items():
        print(f"\nTraining {feature_type}__{name}...")

        start_time = time.time()

        try:
            model.fit(X_train, y_train)

            train_pred = model.predict(X_train)
            val_pred = model.predict(X_val)

            train_acc = accuracy_score(y_train, train_pred)
            val_acc = accuracy_score(y_val, val_pred)

            elapsed = time.time() - start_time

            model_key = f"{feature_type}__{name}"

            results.append({
                "model_name": model_key,
                "base_model": name,
                "feature_type": feature_type,
                "train_accuracy": train_acc,
                "validation_accuracy": val_acc,
                "train_time_sec": elapsed
            })

            fitted_models[model_key] = model

            print(f"{model_key}")
            print(f"Train accuracy: {train_acc:.4f}")
            print(f"Validation accuracy: {val_acc:.4f}")
            print(f"Time: {elapsed:.2f} sec")

        except Exception as e:
            print(f"{feature_type}__{name} failed: {e}")

    results_df = pd.DataFrame(results)

    if len(results_df) > 0:
        results_df = results_df.sort_values(
            "validation_accuracy",
            ascending=False
        ).reset_index(drop=True)

    return results_df, fitted_models

In [36]:
base8_raw_results, base8_raw_fitted = train_and_evaluate_models(
    models=get_base_8_models(),
    X_train=X_train_ml,
    X_val=X_val_ml,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="raw"
)

display(base8_raw_results)


Training raw__kNN...
raw__kNN
Train accuracy: 0.9435
Validation accuracy: 0.9295
Time: 0.77 sec

Training raw__DecisionTree...
raw__DecisionTree
Train accuracy: 0.9633
Validation accuracy: 0.9377
Time: 0.07 sec

Training raw__RandomForest...
raw__RandomForest
Train accuracy: 0.9790
Validation accuracy: 0.9608
Time: 0.95 sec

Training raw__ExtraTrees...
raw__ExtraTrees
Train accuracy: 0.9790
Validation accuracy: 0.9608
Time: 0.84 sec

Training raw__HistGradientBoosting...
raw__HistGradientBoosting
Train accuracy: 0.9745
Validation accuracy: 0.9590
Time: 13.16 sec

Training raw__LogisticRegression...
raw__LogisticRegression
Train accuracy: 0.1943
Validation accuracy: 0.1995
Time: 2.18 sec

Training raw__LinearSVC...
raw__LinearSVC
Train accuracy: 0.1801
Validation accuracy: 0.1858
Time: 0.56 sec

Training raw__MLP...
raw__MLP
Train accuracy: 0.9745
Validation accuracy: 0.9382
Time: 32.53 sec


,model_name,base_model,feature_type,train_accuracy,validation_accuracy,train_time_sec
0,raw__ExtraTrees,ExtraTrees,raw,0.979000,0.960833,0.843785
1,raw__RandomForest,RandomForest,raw,0.979000,0.960833,0.945023
2,raw__HistGradientBoosting,HistGradientBoosting,raw,0.974458,0.959000,13.161721
3,raw__MLP,MLP,raw,0.974542,0.938167,32.527652
4,raw__DecisionTree,DecisionTree,raw,0.963250,0.937667,0.066773
5,raw__kNN,kNN,raw,0.943458,0.929500,0.770020
6,raw__LogisticRegression,LogisticRegression,raw,0.194333,0.199500,2.184223
7,raw__LinearSVC,LinearSVC,raw,0.180125,0.185833,0.563591


In [38]:
base8_row_results, base8_row_fitted = train_and_evaluate_models(
    models=get_base_8_models(),
    X_train=X_train_row,
    X_val=X_val_row,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="row_average"
)

display(base8_row_results)


Training row_average__kNN...
row_average__kNN
Train accuracy: 0.1740
Validation accuracy: 0.1700
Time: 0.24 sec

Training row_average__DecisionTree...
row_average__DecisionTree
Train accuracy: 0.2409
Validation accuracy: 0.2300
Time: 0.01 sec

Training row_average__RandomForest...
row_average__RandomForest
Train accuracy: 0.2409
Validation accuracy: 0.2287
Time: 0.54 sec

Training row_average__ExtraTrees...
row_average__ExtraTrees
Train accuracy: 0.2410
Validation accuracy: 0.2300
Time: 0.43 sec

Training row_average__HistGradientBoosting...
row_average__HistGradientBoosting
Train accuracy: 0.2401
Validation accuracy: 0.2322
Time: 6.45 sec

Training row_average__LogisticRegression...
row_average__LogisticRegression
Train accuracy: 0.1460
Validation accuracy: 0.1458
Time: 1.22 sec

Training row_average__LinearSVC...
row_average__LinearSVC
Train accuracy: 0.1471
Validation accuracy: 0.1473
Time: 0.04 sec

Training row_average__MLP...
row_average__MLP
Train accuracy: 0.2063
Validation ac

,model_name,base_model,feature_type,train_accuracy,validation_accuracy,train_time_sec
0,row_average__HistGradientBoosting,HistGradientBoosting,row_average,0.240083,0.232167,6.451998
1,row_average__DecisionTree,DecisionTree,row_average,0.240917,0.230000,0.012805
2,row_average__ExtraTrees,ExtraTrees,row_average,0.240958,0.230000,0.433114
3,row_average__RandomForest,RandomForest,row_average,0.240917,0.228667,0.535909
4,row_average__MLP,MLP,row_average,0.206333,0.198500,33.850253
5,row_average__kNN,kNN,row_average,0.174000,0.170000,0.236717
6,row_average__LinearSVC,LinearSVC,row_average,0.147125,0.147333,0.037703
7,row_average__LogisticRegression,LogisticRegression,row_average,0.146000,0.145833,1.224252


In [39]:
base_8_svd_models = {}

for name, model in get_base_8_models().items():
    base_8_svd_models[name] = Pipeline([
        ("svd", TruncatedSVD(n_components=5, random_state=RANDOM_STATE)),
        ("model", model)
    ])

base8_svd_results, base8_svd_fitted = train_and_evaluate_models(
    models=base_8_svd_models,
    X_train=X_train_ml,
    X_val=X_val_ml,
    y_train=y_train_ml,
    y_val=y_val_ml,
    feature_type="svd"
)

display(base8_svd_results)


Training svd__kNN...
svd__kNN
Train accuracy: 0.9229
Validation accuracy: 0.9032
Time: 0.61 sec

Training svd__DecisionTree...
svd__DecisionTree
Train accuracy: 0.9318
Validation accuracy: 0.8958
Time: 0.33 sec

Training svd__RandomForest...
svd__RandomForest
Train accuracy: 0.9790
Validation accuracy: 0.9310
Time: 1.84 sec

Training svd__ExtraTrees...
svd__ExtraTrees
Train accuracy: 0.9790
Validation accuracy: 0.9367
Time: 1.12 sec

Training svd__HistGradientBoosting...
svd__HistGradientBoosting
Train accuracy: 0.9711
Validation accuracy: 0.9303
Time: 15.37 sec

Training svd__LogisticRegression...
svd__LogisticRegression
Train accuracy: 0.1511
Validation accuracy: 0.1523
Time: 1.68 sec

Training svd__LinearSVC...
svd__LinearSVC
Train accuracy: 0.1528
Validation accuracy: 0.1508
Time: 0.59 sec

Training svd__MLP...
svd__MLP
Train accuracy: 0.9170
Validation accuracy: 0.8958
Time: 31.85 sec


,model_name,base_model,feature_type,train_accuracy,validation_accuracy,train_time_sec
0,svd__ExtraTrees,ExtraTrees,svd,0.979000,0.936667,1.121237
1,svd__RandomForest,RandomForest,svd,0.979000,0.931000,1.835491
2,svd__HistGradientBoosting,HistGradientBoosting,svd,0.971125,0.930333,15.369794
3,svd__kNN,kNN,svd,0.922875,0.903167,0.610520
4,svd__DecisionTree,DecisionTree,svd,0.931833,0.895833,0.333750
5,svd__MLP,MLP,svd,0.917042,0.895833,31.847713
6,svd__LogisticRegression,LogisticRegression,svd,0.151083,0.152333,1.684198
7,svd__LinearSVC,LinearSVC,svd,0.152792,0.150833,0.592952


In [40]:
top_base8_models_df = base8_all_results.head(TOP_N_MODELS).copy()

print("Selected top models:")
display(top_base8_models_df)

NameError: name 'base8_all_results' is not defined

In [ ]:
class Base8ModelWrapper:
    def __init__(self, model_name, model, feature_type, validation_accuracy):
        self.model_name = model_name
        self.model = model
        self.feature_type = feature_type
        self.validation_accuracy = validation_accuracy

    def transform(self, X):
        if self.feature_type == "raw":
            return X

        if self.feature_type == "row_average":
            return row_average_features(X)

        if self.feature_type == "svd":
            return X

        raise ValueError(f"Unknown feature type: {self.feature_type}")

    def predict(self, X):
        Xt = self.transform(X)
        return self.model.predict(Xt)

    def predict_proba(self, X):
        Xt = self.transform(X)
        return safe_predict_proba(
            self.model,
            Xt,
            num_classes=len(label_encoder.classes_)
        )

top_base8_wrapped_models = []

for _, row in top_base8_models_df.iterrows():
    model_name = row["model_name"]
    feature_type = row["feature_type"]
    model = base8_all_fitted[model_name]

    wrapped = Base8ModelWrapper(
        model_name=model_name,
        model=model,
        feature_type=feature_type,
        validation_accuracy=row["validation_accuracy"]
    )

    top_base8_wrapped_models.append(wrapped)

print("Models used in weighted ensemble:")
for model in top_base8_wrapped_models:
    print(model.model_name, "| acc:", model.validation_accuracy)

In [ ]:
def weighted_ensemble_predict_proba(wrapped_models, X):
    weights = np.array([m.validation_accuracy for m in wrapped_models])
    weights = weights / weights.sum()

    final_proba = None

    for model, weight in zip(wrapped_models, weights):
        proba = model.predict_proba(X)

        if final_proba is None:
            final_proba = weight * proba
        else:
            final_proba += weight * proba

    return final_proba

def weighted_ensemble_predict(wrapped_models, X):
    proba = weighted_ensemble_predict_proba(wrapped_models, X)
    return np.argmax(proba, axis=1)

ensemble_val_pred = weighted_ensemble_predict(
    top_base8_wrapped_models,
    X_val_ml
)

ensemble_val_acc = accuracy_score(y_val_ml, ensemble_val_pred)

print("Best single model accuracy:", base8_all_results.iloc[0]["validation_accuracy"])
print("Weighted ensemble accuracy:", ensemble_val_acc)

In [ ]:
def top_k_accuracy(y_true, proba, k=3):
    top_k = np.argsort(proba, axis=1)[:, -k:]
    correct = 0

    for true_label, candidates in zip(y_true, top_k):
        if true_label in candidates:
            correct += 1

    return correct / len(y_true)

ensemble_val_proba = weighted_ensemble_predict_proba(
    top_base8_wrapped_models,
    X_val_ml
)

for k in [1, 2, 3, 5]:
    acc = top_k_accuracy(y_val_ml, ensemble_val_proba, k=k)
    print(f"Top-{k} accuracy: {acc:.4f}")

In [ ]:
def predict_masked_kmer(masked_kmer):
    x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)

    proba = weighted_ensemble_predict_proba(
        top_base8_wrapped_models,
        x
    )[0]

    pred_id = int(np.argmax(proba))
    pred_aa = label_encoder.inverse_transform([pred_id])[0]
    confidence = float(proba[pred_id])

    top_ids = np.argsort(proba)[::-1][:5]

    top_candidates = []

    for idx in top_ids:
        aa = label_encoder.inverse_transform([idx])[0]
        top_candidates.append({
            "amino_acid": aa,
            "confidence": float(proba[idx])
        })

    return pred_aa, confidence, top_candidates

masked_example = "DIQMTQSPSS-"
pred_aa, conf, top_candidates = predict_masked_kmer(masked_example)

print("Masked k-mer:", masked_example)
print("Prediction:", pred_aa)
print("Confidence:", conf)
print("Top candidates:", top_candidates)

In [ ]:
def fill_known_size_gap_ml(left_context, right_context, gap_size, k=11):
    filled = ""
    residue_confidences = []

    for _ in range(gap_size):
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)

        masked_kmer = prefix + GAP_TOKEN

        pred_aa, conf, _ = predict_masked_kmer(masked_kmer)

        filled += pred_aa
        residue_confidences.append(conf)

    confidence = float(np.mean(residue_confidences)) if residue_confidences else 0.0

    return {
        "prediction": filled,
        "confidence": confidence,
        "mass": peptide_mass(filled)
    }

known_size_example = fill_known_size_gap_ml(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    gap_size=3
)

print(known_size_example)

In [ ]:
def create_gap_sample(sequence, protein_id, gap_start, gap_len, context_size=15):
    gap_end = gap_start + gap_len
    gap_seq = sequence[gap_start:gap_end]

    left_context = sequence[max(0, gap_start - context_size):gap_start]
    right_context = sequence[gap_end:gap_end + context_size]

    return {
        "protein_id": protein_id,
        "full_sequence": sequence,
        "gap_start": gap_start,
        "gap_end": gap_end,
        "gap_size": gap_len,
        "gap_sequence": gap_seq,
        "gap_mass": peptide_mass(gap_seq),
        "left_context": left_context,
        "right_context": right_context
    }

def generate_gap_dataset(
    sequences,
    protein_prefix,
    samples_per_sequence=2,
    min_gap=1,
    max_gap=8,
    context_size=15
):
    rows = []

    for idx, seq in enumerate(sequences):
        if len(seq) < max_gap + 2 * context_size + 5:
            continue

        for _ in range(samples_per_sequence):
            gap_len = random.randint(min_gap, max_gap)

            min_start = context_size
            max_start = len(seq) - gap_len - context_size

            if max_start <= min_start:
                continue

            gap_start = random.randint(min_start, max_start)

            rows.append(
                create_gap_sample(
                    sequence=seq,
                    protein_id=f"{protein_prefix}_{idx}",
                    gap_start=gap_start,
                    gap_len=gap_len,
                    context_size=context_size
                )
            )

    return pd.DataFrame(rows)

gap_df = pd.concat([
    generate_gap_dataset(mab_train_seqs, "MAB", SAMPLES_PER_SEQUENCE),
    generate_gap_dataset(p5a_train_seqs, "P5A", SAMPLES_PER_SEQUENCE),
    generate_gap_dataset(cah2_train_seqs, "CAH2", SAMPLES_PER_SEQUENCE)
], ignore_index=True)

print("Gap evaluation samples:", gap_df.shape)
display(gap_df.head())

In [ ]:
def residue_accuracy(pred, truth):
    if len(pred) == 0 or len(truth) == 0:
        return 0.0

    n = min(len(pred), len(truth))
    matches = sum(pred[i] == truth[i] for i in range(n))
    return matches / max(len(pred), len(truth))

def evaluate_known_size_gap_filling(df, max_rows=200):
    rows = []

    eval_df = df.head(max_rows).copy()

    for _, row in eval_df.iterrows():
        result = fill_known_size_gap_ml(
            left_context=row["left_context"],
            right_context=row["right_context"],
            gap_size=int(row["gap_size"])
        )

        pred = result["prediction"]
        truth = row["gap_sequence"]

        rows.append({
            "protein_id": row["protein_id"],
            "truth": truth,
            "prediction": pred,
            "gap_size": row["gap_size"],
            "truth_mass": row["gap_mass"],
            "pred_mass": result["mass"],
            "confidence": result["confidence"],
            "exact_match": pred == truth,
            "residue_accuracy": residue_accuracy(pred, truth)
        })

    return pd.DataFrame(rows)

known_size_eval_df = evaluate_known_size_gap_filling(
    gap_df,
    max_rows=200
)

display(known_size_eval_df.head(20))

print("Known-size exact match accuracy:", known_size_eval_df["exact_match"].mean())
print("Known-size mean residue accuracy:", known_size_eval_df["residue_accuracy"].mean())

In [ ]:
def generate_mass_candidates_from_homologs(
    homolog_sequences,
    target_mass,
    tolerance=0,
    min_len=None,
    max_len=None
):
    if min_len is None:
        min_len = max(1, math.floor((target_mass - tolerance) / max(AA_MASS.values())))

    if max_len is None:
        max_len = math.ceil((target_mass + tolerance) / min(AA_MASS.values()))

    candidate_counter = Counter()

    for seq in homolog_sequences:
        n = len(seq)

        for length in range(min_len, max_len + 1):
            if n < length:
                continue

            for i in range(n - length + 1):
                kmer = seq[i:i+length]

                if not is_valid_protein_sequence(kmer):
                    continue

                mass = peptide_mass(kmer)

                if abs(mass - target_mass) <= tolerance:
                    candidate_counter[kmer] += 1

    return candidate_counter

def context_support_score(candidate, left_context, right_context, homolog_sequences, flank=5):
    left = left_context[-flank:] if left_context else ""
    right = right_context[:flank] if right_context else ""

    pattern_full = left + candidate + right
    pattern_left = left + candidate
    pattern_right = candidate + right

    full_count = 0
    left_count = 0
    right_count = 0
    candidate_count = 0

    for seq in homolog_sequences:
        full_count += seq.count(pattern_full)
        left_count += seq.count(pattern_left)
        right_count += seq.count(pattern_right)
        candidate_count += seq.count(candidate)

    score = (
        5.0 * full_count +
        2.0 * left_count +
        2.0 * right_count +
        1.0 * candidate_count
    )

    return float(score)

def probabilistic_rank_candidates(
    target_mass,
    left_context,
    right_context,
    homolog_sequences,
    tolerance=0,
    top_k=50
):
    candidates = generate_mass_candidates_from_homologs(
        homolog_sequences=homolog_sequences,
        target_mass=target_mass,
        tolerance=tolerance
    )

    rows = []

    for candidate, frequency in candidates.items():
        mass = peptide_mass(candidate)
        mass_error = abs(mass - target_mass)

        context_score = context_support_score(
            candidate=candidate,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            flank=5
        )

        probabilistic_score = frequency + context_score - mass_error

        rows.append({
            "candidate": candidate,
            "candidate_length": len(candidate),
            "candidate_mass": mass,
            "target_mass": target_mass,
            "mass_error": mass_error,
            "homolog_frequency": frequency,
            "context_score": context_score,
            "probabilistic_score": probabilistic_score,
            "mass_valid": mass_error <= tolerance
        })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        return df

    return df.sort_values(
        "probabilistic_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)

In [41]:
def ml_score_candidate(left_context, right_context, candidate, k=11):
    if len(candidate) == 0:
        return 0.0

    scores = []
    filled = ""

    for aa in candidate:
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)

        masked_kmer = prefix + GAP_TOKEN

        x = np.array(encode_sequence(masked_kmer)).reshape(1, -1)

        proba = weighted_ensemble_predict_proba(
            top_base8_wrapped_models,
            x
        )[0]

        if aa in label_encoder.classes_:
            aa_id = label_encoder.transform([aa])[0]
            scores.append(float(proba[aa_id]))
        else:
            scores.append(0.0)

        filled += aa

    return float(np.mean(scores)) if scores else 0.0

def hybrid_rerank_candidates(
    candidate_df,
    left_context,
    right_context,
    target_mass,
    mass_tolerance=0
):
    if candidate_df is None or len(candidate_df) == 0:
        return pd.DataFrame()

    rows = []

    for _, row in candidate_df.iterrows():
        candidate = row["candidate"]

        ml_score = ml_score_candidate(
            left_context=left_context,
            right_context=right_context,
            candidate=candidate,
            k=11
        )

        candidate_mass = peptide_mass(candidate)
        mass_error = abs(candidate_mass - target_mass)

        mass_valid_score = 1.0 if mass_error <= mass_tolerance else 0.0
        homolog_frequency_score = math.log1p(row["homolog_frequency"])
        context_score_norm = math.log1p(row["context_score"])

        hybrid_score = (
            3.0 * mass_valid_score +
            2.0 * homolog_frequency_score +
            2.0 * context_score_norm +
            2.0 * ml_score -
            2.0 * mass_error
        )

        out = row.to_dict()
        out.update({
            "ml_ensemble_score": ml_score,
            "mass_valid_score": mass_valid_score,
            "hybrid_score": hybrid_score
        })

        rows.append(out)

    out_df = pd.DataFrame(rows)

    return out_df.sort_values(
        "hybrid_score",
        ascending=False
    ).reset_index(drop=True)

In [42]:
cah2_test_cases = [
    {
        "gap_name": "CAH2_gap_1",
        "mass": 420,
        "left": "MSHHWGYGKHN",
        "right": "WHKDFPIAKGER",
        "truth": "GPEH"
    },
    {
        "gap_name": "CAH2_gap_2",
        "mass": 750,
        "left": "DPSLKPL",
        "right": "TSLRILNNGHA",
        "truth": "SVSYDQA"
    },
    {
        "gap_name": "CAH2_gap_3",
        "mass": 622,
        "left": "SQDK",
        "right": "LDGTYRLIQF",
        "truth": "AVLKGGP"
    },
    {
        "gap_name": "CAH2_gap_4",
        "mass": 1318,
        "left": "VHWNT",
        "right": "DGLAVLGIFL",
        "truth": "KYGDFGKAVQQP"
    },
    {
        "gap_name": "CAH2_gap_5",
        "mass": 751,
        "left": "DYWTYPGSL",
        "right": "CVTWIVLKEP",
        "truth": "TTPPLLE"
    },
    {
        "gap_name": "CAH2_gap_6",
        "mass": 544,
        "left": "VSSEQV",
        "right": "KLNFNGEGEP",
        "truth": "LKFR"
    },
    {
        "gap_name": "CAH2_gap_7",
        "mass": 561,
        "left": "PLKNRQI",
        "right": "",
        "truth": "KASFK"
    }
]

In [43]:
known_mass_results = []

for case in cah2_test_cases:
    print("\nRunning:", case["gap_name"], "mass:", case["mass"])

    prob_candidates = probabilistic_rank_candidates(
        target_mass=case["mass"],
        left_context=case["left"],
        right_context=case["right"],
        homolog_sequences=cah2_train_seqs,
        tolerance=0,
        top_k=50
    )

    hybrid_candidates = hybrid_rerank_candidates(
        candidate_df=prob_candidates,
        left_context=case["left"],
        right_context=case["right"],
        target_mass=case["mass"],
        mass_tolerance=0
    )

    if len(hybrid_candidates) == 0:
        pred = ""
        pred_mass = None
        hybrid_score = None
        rank_of_truth = None
        top_5 = []
    else:
        pred = hybrid_candidates.iloc[0]["candidate"]
        pred_mass = hybrid_candidates.iloc[0]["candidate_mass"]
        hybrid_score = hybrid_candidates.iloc[0]["hybrid_score"]
        top_5 = hybrid_candidates["candidate"].head(5).tolist()

        truth_positions = hybrid_candidates.index[
            hybrid_candidates["candidate"] == case["truth"]
        ].tolist()

        rank_of_truth = truth_positions[0] + 1 if truth_positions else None

    known_mass_results.append({
        "gap_name": case["gap_name"],
        "target_mass": case["mass"],
        "truth": case["truth"],
        "prediction": pred,
        "pred_mass": pred_mass,
        "hybrid_score": hybrid_score,
        "exact_match": pred == case["truth"],
        "rank_of_truth": rank_of_truth,
        "top_5_candidates": top_5
    })

    display(hybrid_candidates.head(10))

known_mass_results_df = pd.DataFrame(known_mass_results)

display(known_mass_results_df)

print("Known-mass exact match accuracy:", known_mass_results_df["exact_match"].mean())

known_mass_top5 = known_mass_results_df["rank_of_truth"].apply(
    lambda x: x is not None and x <= 5
).mean()

print("Known-mass top-5 recovery:", known_mass_top5)


Running: CAH2_gap_1 mass: 420


NameError: name 'probabilistic_rank_candidates' is not defined

Cell 1 — Imports, configuration, and paths

In [19]:
import os
import re
import math
import time
import random
import warnings
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD, PCA

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    SGDClassifier,
    PassiveAggressiveClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Main Kaggle dataset path given by you
DATA_DIR = "/kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets"

DE_NOVO_CAH2 = f"{DATA_DIR}/de_novo_sequence_CAH2.txt"
MAB_TARGET = f"{DATA_DIR}/mab_target_sequence.txt"
MAB_TRAINING = f"{DATA_DIR}/mab_training_sequence.txt"
P5A_TRAINING = f"{DATA_DIR}/p5A_training_sequence.txt"
CAH2_TARGET = f"{DATA_DIR}/target_sequence_CAH2.txt"
CAH2_TRAINING = f"{DATA_DIR}/training_sequences_CAH2.txt"

# Safety fallback: auto-find files if Kaggle path is slightly different
def find_file(filename, root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

required_files = {
    "DE_NOVO_CAH2": ("de_novo_sequence_CAH2.txt", DE_NOVO_CAH2),
    "MAB_TARGET": ("mab_target_sequence.txt", MAB_TARGET),
    "MAB_TRAINING": ("mab_training_sequence.txt", MAB_TRAINING),
    "P5A_TRAINING": ("p5A_training_sequence.txt", P5A_TRAINING),
    "CAH2_TARGET": ("target_sequence_CAH2.txt", CAH2_TARGET),
    "CAH2_TRAINING": ("training_sequences_CAH2.txt", CAH2_TRAINING),
}

fixed_paths = {}

for key, (fname, path) in required_files.items():
    if os.path.exists(path):
        fixed_paths[key] = path
    else:
        found = find_file(fname)
        fixed_paths[key] = found
        print(f"Auto-found {key}: {found}")

DE_NOVO_CAH2 = fixed_paths["DE_NOVO_CAH2"]
MAB_TARGET = fixed_paths["MAB_TARGET"]
MAB_TRAINING = fixed_paths["MAB_TRAINING"]
P5A_TRAINING = fixed_paths["P5A_TRAINING"]
CAH2_TARGET = fixed_paths["CAH2_TARGET"]
CAH2_TRAINING = fixed_paths["CAH2_TRAINING"]

for name, path in fixed_paths.items():
    print(name, "=>", path, "| exists:", path is not None and os.path.exists(path))

DE_NOVO_CAH2 => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/de_novo_sequence_CAH2.txt | exists: True
MAB_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_target_sequence.txt | exists: True
MAB_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/mab_training_sequence.txt | exists: True
P5A_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/p5A_training_sequence.txt | exists: True
CAH2_TARGET => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/target_sequence_CAH2.txt | exists: True
CAH2_TRAINING => /kaggle/input/datasets/tahmidenamshrestha/protei-scaffold-datasets/training_sequences_CAH2.txt | exists: True


Cell 2 — Amino acid constants and mass table

In [20]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
GAP_TOKEN = "-"
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Integer masses used in the paper-style implementation
AA_MASS = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99,
    "T": 101, "C": 103, "L": 113, "I": 113, "N": 114,
    "D": 115, "Q": 128, "K": 128, "E": 129, "M": 131,
    "H": 137, "F": 147, "R": 156, "Y": 163, "W": 186
}

aa_to_int = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa_to_int[GAP_TOKEN] = 0

int_to_aa = {v: k for k, v in aa_to_int.items()}

def peptide_mass(seq):
    return int(sum(AA_MASS.get(aa, 0) for aa in seq))

def is_valid_protein_sequence(seq):
    return all(ch in AMINO_ACIDS for ch in seq)

print("Amino acid vocabulary:", aa_to_int)
print("Mass of GPEH:", peptide_mass("GPEH"))
print("Mass of SVSYDQA:", peptide_mass("SVSYDQA"))

Amino acid vocabulary: {'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20, '-': 0}
Mass of GPEH: 420
Mass of SVSYDQA: 750


Cell 3 — FASTA/text parser

In [21]:
def read_text(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def clean_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    return seq

def parse_fasta_or_sequences(path):
    """
    Parses FASTA-like or plain protein sequence files.
    Returns list of cleaned protein sequences.
    """
    text = read_text(path)
    sequences = []
    current = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        if line.startswith(">"):
            if current:
                seq = clean_sequence("".join(current))
                if seq:
                    sequences.append(seq)
                current = []
        else:
            current.append(line)

    if current:
        seq = clean_sequence("".join(current))
        if seq:
            sequences.append(seq)

    return sequences

def encode_sequence(seq):
    return [aa_to_int.get(ch, 0) for ch in seq]

mab_target_seq = parse_fasta_or_sequences(MAB_TARGET)[0]
mab_train_seqs = parse_fasta_or_sequences(MAB_TRAINING)

p5a_train_seqs = parse_fasta_or_sequences(P5A_TRAINING)

cah2_target_seq = parse_fasta_or_sequences(CAH2_TARGET)[0]
cah2_train_seqs = parse_fasta_or_sequences(CAH2_TRAINING)

all_homologs = mab_train_seqs + p5a_train_seqs + cah2_train_seqs

print("Mab target length:", len(mab_target_seq))
print("Mab training sequences:", len(mab_train_seqs))
print("P5A training sequences:", len(p5a_train_seqs))
print("CAH2 target length:", len(cah2_target_seq))
print("CAH2 training sequences:", len(cah2_train_seqs))
print("All homolog sequences:", len(all_homologs))

Mab target length: 214
Mab training sequences: 992
P5A training sequences: 1000
CAH2 target length: 260
CAH2 training sequences: 1000
All homolog sequences: 2992


Cell 4 — Generate scaffold gap samples from real protein sequences

In [22]:
def create_gap_sample(
    sequence,
    protein_id,
    gap_start,
    gap_len,
    context_size=15
):
    gap_end = gap_start + gap_len
    gap_seq = sequence[gap_start:gap_end]

    left_context = sequence[max(0, gap_start - context_size):gap_start]
    right_context = sequence[gap_end:gap_end + context_size]

    scaffold_known_size = (
        sequence[:gap_start] +
        (GAP_TOKEN * gap_len) +
        sequence[gap_end:]
    )

    scaffold_known_mass = (
        sequence[:gap_start] +
        "{" + str(peptide_mass(gap_seq)) + " Da}" +
        sequence[gap_end:]
    )

    return {
        "protein_id": protein_id,
        "full_sequence": sequence,
        "gap_start": gap_start,
        "gap_end": gap_end,
        "gap_size": gap_len,
        "gap_sequence": gap_seq,
        "gap_mass": peptide_mass(gap_seq),
        "left_context": left_context,
        "right_context": right_context,
        "scaffold_known_size": scaffold_known_size,
        "scaffold_known_mass": scaffold_known_mass
    }

def generate_gap_dataset(
    sequences,
    protein_prefix,
    samples_per_sequence=5,
    min_gap=1,
    max_gap=12,
    context_size=15
):
    rows = []

    for idx, seq in enumerate(sequences):
        if len(seq) < (max_gap + 2 * context_size + 5):
            continue

        for _ in range(samples_per_sequence):
            gap_len = random.randint(min_gap, max_gap)

            min_start = context_size
            max_start = len(seq) - gap_len - context_size

            if max_start <= min_start:
                continue

            gap_start = random.randint(min_start, max_start)

            sample = create_gap_sample(
                sequence=seq,
                protein_id=f"{protein_prefix}_{idx}",
                gap_start=gap_start,
                gap_len=gap_len,
                context_size=context_size
            )

            rows.append(sample)

    return pd.DataFrame(rows)

# Generate simulated scaffold gap samples from real homologous protein sequences
gap_df = pd.concat([
    generate_gap_dataset(mab_train_seqs, "MAB", samples_per_sequence=5, min_gap=1, max_gap=12),
    generate_gap_dataset(p5a_train_seqs, "P5A", samples_per_sequence=5, min_gap=1, max_gap=12),
    generate_gap_dataset(cah2_train_seqs, "CAH2", samples_per_sequence=5, min_gap=1, max_gap=12),
], ignore_index=True)

print("Generated gap samples:", gap_df.shape)
display(gap_df.head())

Generated gap samples: (14960, 11)


,protein_id,full_sequence,gap_start,gap_end,gap_size,gap_sequence,gap_mass,left_context,right_context,scaffold_known_size,scaffold_known_mass
0,MAB_0,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,43,54,11,PKLLIYNTNNL,1283,IDKYLNWYQQKPGKA,QTGVPSRFSGSGSGT,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKA---...,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKA{12...
1,MAB_0,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,85,86,1,Y,163,FTFTISSLQPEDIAT,YCLQHISRPRTFGQG,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...
2,MAB_0,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,72,76,4,FTIS,448,VPSRFSGSGSGTDFT,SLQPEDIATYYCLQH,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...
3,MAB_0,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,41,44,3,KAP,296,QNIDKYLNWYQQKPG,KLLIYNTNNLQTGVP,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPG---KL...,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPG{296 ...
4,MAB_0,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,154,165,11,QSGNSQESVTE,1146,YPREAKVQWKVDNAL,QDSKDSTYSLSSTLT,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...


Cell 5 — Train/validation/test split by protein ID

In [23]:
unique_proteins = gap_df["protein_id"].unique()

train_proteins, temp_proteins = train_test_split(
    unique_proteins,
    test_size=0.30,
    random_state=RANDOM_STATE
)

val_proteins, test_proteins = train_test_split(
    temp_proteins,
    test_size=0.50,
    random_state=RANDOM_STATE
)

train_gap_df = gap_df[gap_df["protein_id"].isin(train_proteins)].reset_index(drop=True)
val_gap_df = gap_df[gap_df["protein_id"].isin(val_proteins)].reset_index(drop=True)
test_gap_df = gap_df[gap_df["protein_id"].isin(test_proteins)].reset_index(drop=True)

print("Train gap samples:", train_gap_df.shape)
print("Validation gap samples:", val_gap_df.shape)
print("Test gap samples:", test_gap_df.shape)

Train gap samples: (10470, 11)
Validation gap samples: (2245, 11)
Test gap samples: (2245, 11)


Cell 6 — Known-gap-size ML dataset using 11-mer masking

In [24]:
def generate_masked_kmer_dataset(sequences, k=11, max_samples=None):
    X = []
    y_letters = []

    for seq in sequences:
        if len(seq) < k:
            continue

        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]

            if not is_valid_protein_sequence(kmer):
                continue

            # Mask first amino acid
            masked_first = GAP_TOKEN + kmer[1:]
            X.append(encode_sequence(masked_first))
            y_letters.append(kmer[0])

            # Mask last amino acid
            masked_last = kmer[:-1] + GAP_TOKEN
            X.append(encode_sequence(masked_last))
            y_letters.append(kmer[-1])

            if max_samples is not None and len(X) >= max_samples:
                return np.array(X), np.array(y_letters)

    return np.array(X), np.array(y_letters)

# Use all homologs for residue-level ML training
X_ml, y_ml_letters = generate_masked_kmer_dataset(
    all_homologs,
    k=11,
    max_samples=None
)

label_encoder = LabelEncoder()
y_ml = label_encoder.fit_transform(y_ml_letters)

X_train_ml, X_val_ml, y_train_ml, y_val_ml = train_test_split(
    X_ml,
    y_ml,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_ml
)

print("X_ml:", X_ml.shape)
print("Classes:", list(label_encoder.classes_))
print("Train:", X_train_ml.shape)
print("Val:", X_val_ml.shape)

X_ml: (1344294, 11)
Classes: [np.str_('A'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('V'), np.str_('W'), np.str_('Y')]
Train: (1075435, 11)
Val: (268859, 11)


Cell 7 — Feature engineering helpers

In [25]:
def row_average_features(X):
    return X.mean(axis=1).reshape(-1, 1)

def aa_frequency_features(X):
    """
    Frequency of each encoded amino acid value in a k-mer.
    Includes gap token 0 plus amino acids 1..20.
    """
    features = []

    for row in X:
        counts = np.zeros(21)
        for val in row:
            if 0 <= int(val) <= 20:
                counts[int(val)] += 1
        counts = counts / len(row)
        features.append(counts)

    return np.array(features)

def physicochemical_features_from_encoded(X):
    """
    Simple numerical descriptors from amino-acid masses in encoded k-mer.
    Gap token gets mass 0.
    """
    int_to_mass = {0: 0}
    for aa, idx in aa_to_int.items():
        if aa in AA_MASS:
            int_to_mass[idx] = AA_MASS[aa]

    features = []

    for row in X:
        masses = np.array([int_to_mass.get(int(v), 0) for v in row])
        features.append([
            masses.mean(),
            masses.std(),
            masses.min(),
            masses.max(),
            masses.sum(),
            np.count_nonzero(masses)
        ])

    return np.array(features)

def combined_engineered_features(X):
    return np.hstack([
        X,
        row_average_features(X),
        aa_frequency_features(X),
        physicochemical_features_from_encoded(X)
    ])

X_train_row = row_average_features(X_train_ml)
X_val_row = row_average_features(X_val_ml)

X_train_eng = combined_engineered_features(X_train_ml)
X_val_eng = combined_engineered_features(X_val_ml)

print("Raw:", X_train_ml.shape)
print("Row average:", X_train_row.shape)
print("Engineered:", X_train_eng.shape)

Raw: (1075435, 11)
Row average: (1075435, 1)
Engineered: (1075435, 39)


Cell 8 — Optional libraries: XGBoost, LightGBM, CatBoost

In [26]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost unavailable:", e)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM unavailable:", e)

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception as e:
    HAS_CATBOOST = False
    print("CatBoost unavailable:", e)

print("HAS_XGB:", HAS_XGB)
print("HAS_LGBM:", HAS_LGBM)
print("HAS_CATBOOST:", HAS_CATBOOST)

HAS_XGB: True
HAS_LGBM: True
HAS_CATBOOST: True


Cell 9 — Define all ML models

In [27]:
def make_bagging_tree():
    try:
        return BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE),
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    except TypeError:
        return BaggingClassifier(
            base_estimator=DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE),
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

def get_model_bank(num_classes):
    models = {}

    models["kNN"] = KNeighborsClassifier(n_neighbors=5)

    models["DecisionTree"] = DecisionTreeClassifier(
        max_depth=20,
        min_samples_split=5,
        random_state=RANDOM_STATE
    )

    models["ExtraTree"] = ExtraTreeClassifier(
        random_state=RANDOM_STATE
    )

    models["RandomForest"] = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    models["ExtraTrees"] = ExtraTreesClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    models["BaggingTree"] = make_bagging_tree()

    models["GradientBoosting"] = GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )

    models["AdaBoost"] = AdaBoostClassifier(
        random_state=RANDOM_STATE
    )

    models["HistGradientBoosting"] = HistGradientBoostingClassifier(
        random_state=RANDOM_STATE
    )

    models["LogisticRegression"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, n_jobs=-1))
    ])

    models["RidgeClassifier"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RidgeClassifier())
    ])

    models["SGDClassifier"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SGDClassifier(loss="log_loss", max_iter=1000, random_state=RANDOM_STATE))
    ])

    models["PassiveAggressive"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", PassiveAggressiveClassifier(max_iter=1000, random_state=RANDOM_STATE))
    ])

    models["GaussianNB"] = GaussianNB()

    models["LinearSVC"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(random_state=RANDOM_STATE))
    ])

    models["SVC_RBF"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ])

    models["MLP"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(150, 75),
            activation="tanh",
            max_iter=300,
            random_state=RANDOM_STATE
        ))
    ])

    models["LDA"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis())
    ])

    models["QDA"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", QuadraticDiscriminantAnalysis())
    ])

    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            num_class=num_classes,
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    if HAS_LGBM:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        )

    if HAS_CATBOOST:
        models["CatBoost"] = CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="MultiClass",
            verbose=False,
            random_state=RANDOM_STATE
        )

    return models

base_models = get_model_bank(num_classes=len(label_encoder.classes_))
print("Base models:", len(base_models))
print(list(base_models.keys()))

Base models: 22
['kNN', 'DecisionTree', 'ExtraTree', 'RandomForest', 'ExtraTrees', 'BaggingTree', 'GradientBoosting', 'AdaBoost', 'HistGradientBoosting', 'LogisticRegression', 'RidgeClassifier', 'SGDClassifier', 'PassiveAggressive', 'GaussianNB', 'LinearSVC', 'SVC_RBF', 'MLP', 'LDA', 'QDA', 'XGBoost', 'LightGBM', 'CatBoost']


Cell 10 — Train all model variants

In [28]:
def safe_predict_proba(model, X, num_classes):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        if scores.ndim == 1:
            scores = np.vstack([-scores, scores]).T

        scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    preds = model.predict(X)
    proba = np.zeros((len(preds), num_classes))
    for i, p in enumerate(preds):
        proba[i, int(p)] = 1.0
    return proba

def evaluate_model_set(model_bank, X_train, X_val, y_train, y_val, feature_name):
    results = []
    fitted = {}

    for name, model in model_bank.items():
        full_name = f"{feature_name}__{name}"
        print(f"Training {full_name}...")

        start = time.time()

        try:
            model.fit(X_train, y_train)

            train_pred = model.predict(X_train)
            val_pred = model.predict(X_val)

            train_acc = accuracy_score(y_train, train_pred)
            val_acc = accuracy_score(y_val, val_pred)

            elapsed = time.time() - start

            results.append({
                "model_name": full_name,
                "base_model": name,
                "feature_type": feature_name,
                "train_accuracy": train_acc,
                "validation_accuracy": val_acc,
                "train_time_sec": elapsed
            })

            fitted[full_name] = model

            print(f"{full_name}: train={train_acc:.4f}, val={val_acc:.4f}, time={elapsed:.1f}s")

        except Exception as e:
            print(f"{full_name} failed: {e}")

    return pd.DataFrame(results), fitted

all_results = []
all_fitted_models = {}

# 1. Raw encoded features
res_raw, fit_raw = evaluate_model_set(
    base_models,
    X_train_ml,
    X_val_ml,
    y_train_ml,
    y_val_ml,
    "raw"
)
all_results.append(res_raw)
all_fitted_models.update(fit_raw)

# 2. Row average features
row_models = get_model_bank(num_classes=len(label_encoder.classes_))
res_row, fit_row = evaluate_model_set(
    row_models,
    X_train_row,
    X_val_row,
    y_train_ml,
    y_val_ml,
    "row_average"
)
all_results.append(res_row)
all_fitted_models.update(fit_row)

# 3. Engineered features
eng_models = get_model_bank(num_classes=len(label_encoder.classes_))
res_eng, fit_eng = evaluate_model_set(
    eng_models,
    X_train_eng,
    X_val_eng,
    y_train_ml,
    y_val_ml,
    "engineered"
)
all_results.append(res_eng)
all_fitted_models.update(fit_eng)

# 4. SVD pipelines on raw features
svd_models = {}
for name, model in get_model_bank(num_classes=len(label_encoder.classes_)).items():
    svd_models[name] = Pipeline([
        ("svd", TruncatedSVD(n_components=5, random_state=RANDOM_STATE)),
        ("model", model)
    ])

res_svd, fit_svd = evaluate_model_set(
    svd_models,
    X_train_ml,
    X_val_ml,
    y_train_ml,
    y_val_ml,
    "svd"
)
all_results.append(res_svd)
all_fitted_models.update(fit_svd)

# 5. PCA pipelines on raw features
pca_models = {}
for name, model in get_model_bank(num_classes=len(label_encoder.classes_)).items():
    pca_models[name] = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=5, random_state=RANDOM_STATE)),
        ("model", model)
    ])

res_pca, fit_pca = evaluate_model_set(
    pca_models,
    X_train_ml,
    X_val_ml,
    y_train_ml,
    y_val_ml,
    "pca"
)
all_results.append(res_pca)
all_fitted_models.update(fit_pca)

results_df = pd.concat(all_results, ignore_index=True)
results_df = results_df.sort_values("validation_accuracy", ascending=False).reset_index(drop=True)

display(results_df.head(30))

Training raw__kNN...
raw__kNN: train=0.9242, val=0.9088, time=192.7s
Training raw__DecisionTree...
raw__DecisionTree: train=0.8737, val=0.8563, time=6.0s
Training raw__ExtraTree...
raw__ExtraTree: train=0.9617, val=0.9200, time=1.7s
Training raw__RandomForest...
raw__RandomForest: train=0.9617, val=0.9356, time=241.5s
Training raw__ExtraTrees...
raw__ExtraTrees: train=0.9617, val=0.9353, time=197.5s
Training raw__BaggingTree...
raw__BaggingTree: train=0.9310, val=0.9088, time=234.5s
Training raw__GradientBoosting...


KeyboardInterrupt: 

Cell 11 — Select best ML models and create weighted ensemble

In [ ]:
TOP_N_MODELS = 7

top_models_df = results_df.head(TOP_N_MODELS).copy()
display(top_models_df)

def transform_features_for_model(X, feature_type):
    if feature_type == "raw":
        return X
    if feature_type == "row_average":
        return row_average_features(X)
    if feature_type == "engineered":
        return combined_engineered_features(X)
    if feature_type in ["svd", "pca"]:
        return X
    raise ValueError(f"Unknown feature_type: {feature_type}")

class HybridMLModelWrapper:
    def __init__(self, model_name, model, feature_type, validation_accuracy):
        self.model_name = model_name
        self.model = model
        self.feature_type = feature_type
        self.validation_accuracy = validation_accuracy

    def transform(self, X):
        return transform_features_for_model(X, self.feature_type)

    def predict(self, X):
        Xt = self.transform(X)
        return self.model.predict(Xt)

    def predict_proba(self, X):
        Xt = self.transform(X)
        return safe_predict_proba(
            self.model,
            Xt,
            num_classes=len(label_encoder.classes_)
        )

top_wrapped_models = []

for _, row in top_models_df.iterrows():
    model_name = row["model_name"]
    feature_type = row["feature_type"]
    model = all_fitted_models[model_name]

    top_wrapped_models.append(
        HybridMLModelWrapper(
            model_name=model_name,
            model=model,
            feature_type=feature_type,
            validation_accuracy=row["validation_accuracy"]
        )
    )

print("Selected top models:")
for model in top_wrapped_models:
    print(model.model_name, "|", model.feature_type, "|", model.validation_accuracy)

def weighted_ensemble_predict_proba(wrapped_models, X):
    weights = np.array([m.validation_accuracy for m in wrapped_models])
    weights = weights / weights.sum()

    final_proba = None

    for model, w in zip(wrapped_models, weights):
        proba = model.predict_proba(X)

        if final_proba is None:
            final_proba = w * proba
        else:
            final_proba += w * proba

    return final_proba

def weighted_ensemble_predict(wrapped_models, X):
    proba = weighted_ensemble_predict_proba(wrapped_models, X)
    return np.argmax(proba, axis=1)

ensemble_val_pred = weighted_ensemble_predict(top_wrapped_models, X_val_ml)
ensemble_val_acc = accuracy_score(y_val_ml, ensemble_val_pred)

print("Best single ML accuracy:", results_df.iloc[0]["validation_accuracy"])
print("Weighted ensemble accuracy:", ensemble_val_acc)

Cell 12 — Top-k accuracy for ML ensemble

In [ ]:
def top_k_accuracy(y_true, proba, k=3):
    top_k = np.argsort(proba, axis=1)[:, -k:]
    correct = sum(true in candidates for true, candidates in zip(y_true, top_k))
    return correct / len(y_true)

ensemble_val_proba = weighted_ensemble_predict_proba(top_wrapped_models, X_val_ml)

for k in [1, 2, 3, 5]:
    print(f"ML ensemble top-{k} accuracy:", top_k_accuracy(y_val_ml, ensemble_val_proba, k=k))

Cell 13 — ML confidence for candidate gap sequence

In [ ]:
def predict_masked_kmer_ml(masked_kmer):
    X = np.array(encode_sequence(masked_kmer)).reshape(1, -1)
    proba = weighted_ensemble_predict_proba(top_wrapped_models, X)[0]

    pred_id = int(np.argmax(proba))
    pred_aa = label_encoder.inverse_transform([pred_id])[0]
    confidence = float(proba[pred_id])

    return pred_aa, confidence, proba

def ml_score_candidate(left_context, right_context, candidate, k=11):
    """
    Scores a full candidate sequence by average probability of each residue
    under the ML ensemble, generated left-to-right.
    """
    if len(candidate) == 0:
        return 0.0

    probs = []
    filled = ""

    for aa in candidate:
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)
        masked_kmer = prefix + GAP_TOKEN

        _, _, proba = predict_masked_kmer_ml(masked_kmer)

        if aa in label_encoder.classes_:
            aa_id = label_encoder.transform([aa])[0]
            probs.append(float(proba[aa_id]))
        else:
            probs.append(0.0)

        filled += aa

    return float(np.mean(probs))

print("Example masked prediction:")
masked = "DIQMTQSPSS-"
pred, conf, _ = predict_masked_kmer_ml(masked)
print(masked, "=>", pred, conf)

Cell 14 — Iterative known-gap-size filling using ML ensemble

In [ ]:
def fill_known_size_gap_ml(left_context, right_context, gap_size, k=11):
    filled = ""
    residue_confidences = []

    for _ in range(gap_size):
        prefix = (left_context + filled)[-(k-1):]
        prefix = prefix.rjust(k-1, GAP_TOKEN)
        masked_kmer = prefix + GAP_TOKEN

        pred_aa, conf, _ = predict_masked_kmer_ml(masked_kmer)

        filled += pred_aa
        residue_confidences.append(conf)

    confidence = float(np.mean(residue_confidences)) if residue_confidences else 0.0

    return {
        "prediction": filled,
        "confidence": confidence,
        "mass": peptide_mass(filled)
    }

example = fill_known_size_gap_ml(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    gap_size=3
)

print(example)

Cell 15 — Evaluate known-gap-size gap filling on test gap samples

In [ ]:
def residue_accuracy(pred, truth):
    if len(pred) == 0 or len(truth) == 0:
        return 0.0

    n = min(len(pred), len(truth))
    matches = sum(pred[i] == truth[i] for i in range(n))
    return matches / max(len(pred), len(truth))

def evaluate_known_size_gap_filling(df, max_rows=None):
    rows = []

    eval_df = df.copy()

    if max_rows is not None:
        eval_df = eval_df.head(max_rows)

    for _, row in eval_df.iterrows():
        result = fill_known_size_gap_ml(
            left_context=row["left_context"],
            right_context=row["right_context"],
            gap_size=int(row["gap_size"])
        )

        pred = result["prediction"]
        truth = row["gap_sequence"]

        rows.append({
            "protein_id": row["protein_id"],
            "truth": truth,
            "prediction": pred,
            "gap_size": row["gap_size"],
            "truth_mass": row["gap_mass"],
            "pred_mass": result["mass"],
            "confidence": result["confidence"],
            "exact_match": pred == truth,
            "residue_accuracy": residue_accuracy(pred, truth)
        })

    return pd.DataFrame(rows)

known_size_eval_df = evaluate_known_size_gap_filling(test_gap_df, max_rows=200)

display(known_size_eval_df.head(20))

print("Known-size exact match accuracy:", known_size_eval_df["exact_match"].mean())
print("Known-size mean residue accuracy:", known_size_eval_df["residue_accuracy"].mean())

Cell 16 — Deep learning branch: BiLSTM gap sequence model

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (
        Input, Embedding, Bidirectional, LSTM, Dense,
        Dropout, Concatenate, RepeatVector, TimeDistributed
    )
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.callbacks import EarlyStopping

    HAS_TF = True
    print("TensorFlow version:", tf.__version__)
except Exception as e:
    HAS_TF = False
    print("TensorFlow unavailable:", e)

In [ ]:
MAX_CONTEXT = 30
MAX_GAP = 12

dl_vocab = {PAD_TOKEN: 0}
for aa in AMINO_ACIDS:
    dl_vocab[aa] = len(dl_vocab)
dl_vocab[UNK_TOKEN] = len(dl_vocab)

dl_int_to_aa = {v: k for k, v in dl_vocab.items()}

def dl_encode(seq):
    return [dl_vocab.get(ch, dl_vocab[UNK_TOKEN]) for ch in seq]

def prepare_dl_data(df):
    left_inputs = []
    right_inputs = []
    y_outputs = []

    for _, row in df.iterrows():
        gap = row["gap_sequence"]

        if len(gap) > MAX_GAP:
            continue

        left = row["left_context"][-MAX_CONTEXT:]
        right = row["right_context"][:MAX_CONTEXT]

        left_inputs.append(dl_encode(left))
        right_inputs.append(dl_encode(right))
        y_outputs.append(dl_encode(gap))

    left_inputs = pad_sequences(left_inputs, maxlen=MAX_CONTEXT, padding="pre", value=0)
    right_inputs = pad_sequences(right_inputs, maxlen=MAX_CONTEXT, padding="post", value=0)
    y_outputs = pad_sequences(y_outputs, maxlen=MAX_GAP, padding="post", value=0)

    return left_inputs, right_inputs, y_outputs

if HAS_TF:
    dl_X_left_train, dl_X_right_train, dl_y_train = prepare_dl_data(train_gap_df)
    dl_X_left_val, dl_X_right_val, dl_y_val = prepare_dl_data(val_gap_df)

    print("DL left train:", dl_X_left_train.shape)
    print("DL right train:", dl_X_right_train.shape)
    print("DL y train:", dl_y_train.shape)

In [ ]:
def build_bilstm_gap_model(vocab_size, max_context=30, max_gap=12, embed_dim=32):
    left_input = Input(shape=(max_context,), name="left_context")
    right_input = Input(shape=(max_context,), name="right_context")

    embedding = Embedding(
        input_dim=vocab_size,
        output_dim=embed_dim,
        mask_zero=True
    )

    left_emb = embedding(left_input)
    right_emb = embedding(right_input)

    left_encoded = Bidirectional(LSTM(64))(left_emb)
    right_encoded = Bidirectional(LSTM(64))(right_emb)

    merged = Concatenate()([left_encoded, right_encoded])
    merged = Dropout(0.30)(merged)

    repeated = RepeatVector(max_gap)(merged)

    decoded = LSTM(128, return_sequences=True)(repeated)
    decoded = Dropout(0.30)(decoded)

    output = TimeDistributed(
        Dense(vocab_size, activation="softmax"),
        name="gap_output"
    )(decoded)

    model = Model(inputs=[left_input, right_input], outputs=output)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

if HAS_TF:
    bilstm_model = build_bilstm_gap_model(
        vocab_size=len(dl_vocab),
        max_context=MAX_CONTEXT,
        max_gap=MAX_GAP
    )

    bilstm_model.summary()

In [ ]:
if HAS_TF:
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]

    history = bilstm_model.fit(
        [dl_X_left_train, dl_X_right_train],
        dl_y_train[..., None],
        validation_data=([dl_X_left_val, dl_X_right_val], dl_y_val[..., None]),
        epochs=30,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )

Cell 17 — Deep model scoring and prediction

In [ ]:
def dl_predict_gap(left_context, right_context, max_gap=MAX_GAP):
    if not HAS_TF:
        return {
            "prediction": "",
            "confidence": 0.0
        }

    left = pad_sequences(
        [dl_encode(left_context[-MAX_CONTEXT:])],
        maxlen=MAX_CONTEXT,
        padding="pre",
        value=0
    )

    right = pad_sequences(
        [dl_encode(right_context[:MAX_CONTEXT])],
        maxlen=MAX_CONTEXT,
        padding="post",
        value=0
    )

    proba = bilstm_model.predict([left, right], verbose=0)[0]

    pred_ids = np.argmax(proba, axis=1)
    confs = np.max(proba, axis=1)

    pred_chars = []
    pred_confs = []

    for idx, conf in zip(pred_ids, confs):
        aa = dl_int_to_aa.get(int(idx), PAD_TOKEN)

        if aa in [PAD_TOKEN, UNK_TOKEN]:
            continue

        pred_chars.append(aa)
        pred_confs.append(float(conf))

    return {
        "prediction": "".join(pred_chars[:max_gap]),
        "confidence": float(np.mean(pred_confs)) if pred_confs else 0.0
    }

def dl_score_candidate(left_context, right_context, candidate):
    if not HAS_TF or len(candidate) == 0:
        return 0.0

    left = pad_sequences(
        [dl_encode(left_context[-MAX_CONTEXT:])],
        maxlen=MAX_CONTEXT,
        padding="pre",
        value=0
    )

    right = pad_sequences(
        [dl_encode(right_context[:MAX_CONTEXT])],
        maxlen=MAX_CONTEXT,
        padding="post",
        value=0
    )

    proba = bilstm_model.predict([left, right], verbose=0)[0]

    scores = []

    for i, aa in enumerate(candidate[:MAX_GAP]):
        aa_id = dl_vocab.get(aa, dl_vocab[UNK_TOKEN])
        scores.append(float(proba[i, aa_id]))

    return float(np.mean(scores)) if scores else 0.0

if HAS_TF:
    test_dl = dl_predict_gap("DIQMTQSPSSL", "SASVGDRVTIT")
    print(test_dl)

Cell 18 — Known-mass scaffold parser

In [ ]:
def parse_mass_annotated_scaffold_text(text):
    """
    Parses scaffold text containing mass gaps like {420 Da}.
    Returns scaffold pieces and masses.
    """
    pattern = r"\{(\d+)\s*Da\}"

    masses = [int(x) for x in re.findall(pattern, text)]
    pieces = re.split(pattern, text)

    return masses, pieces

de_novo_text = read_text(DE_NOVO_CAH2)

# Extract mass values from CAH2 de novo scaffold file
mass_values = [int(x) for x in re.findall(r"\{(\d+)\s*Da\}", de_novo_text)]

print("Mass gaps found:", mass_values)

Cell 19 — Probabilistic mass candidate generation from homologous sequences

In [ ]:
def generate_mass_candidates_from_homologs(
    homolog_sequences,
    target_mass,
    tolerance=0,
    min_len=None,
    max_len=None
):
    """
    Generate candidate kmers from homologous sequences whose peptide mass
    matches target mass within tolerance.
    """
    if min_len is None:
        min_len = max(1, math.floor((target_mass - tolerance) / max(AA_MASS.values())))

    if max_len is None:
        max_len = math.ceil((target_mass + tolerance) / min(AA_MASS.values()))

    candidate_counter = Counter()

    for seq in homolog_sequences:
        n = len(seq)

        for length in range(min_len, max_len + 1):
            if n < length:
                continue

            for i in range(n - length + 1):
                kmer = seq[i:i+length]

                if not is_valid_protein_sequence(kmer):
                    continue

                mass = peptide_mass(kmer)

                if abs(mass - target_mass) <= tolerance:
                    candidate_counter[kmer] += 1

    return candidate_counter

def context_support_score(candidate, left_context, right_context, homolog_sequences, flank=5):
    left = left_context[-flank:] if left_context else ""
    right = right_context[:flank] if right_context else ""

    pattern_full = left + candidate + right
    pattern_left = left + candidate
    pattern_right = candidate + right

    full_count = 0
    left_count = 0
    right_count = 0
    candidate_count = 0

    for seq in homolog_sequences:
        full_count += seq.count(pattern_full)
        left_count += seq.count(pattern_left)
        right_count += seq.count(pattern_right)
        candidate_count += seq.count(candidate)

    score = (
        5.0 * full_count +
        2.0 * left_count +
        2.0 * right_count +
        1.0 * candidate_count
    )

    return float(score)

def probabilistic_rank_candidates(
    target_mass,
    left_context,
    right_context,
    homolog_sequences,
    tolerance=0,
    top_k=50
):
    candidates = generate_mass_candidates_from_homologs(
        homolog_sequences=homolog_sequences,
        target_mass=target_mass,
        tolerance=tolerance
    )

    rows = []

    for candidate, frequency in candidates.items():
        mass = peptide_mass(candidate)
        mass_error = abs(mass - target_mass)

        context_score = context_support_score(
            candidate=candidate,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            flank=5
        )

        alpha_score = frequency + context_score
        beta_score = mass_error

        prob_score = alpha_score - beta_score

        rows.append({
            "candidate": candidate,
            "candidate_length": len(candidate),
            "candidate_mass": mass,
            "target_mass": target_mass,
            "mass_error": mass_error,
            "homolog_frequency": frequency,
            "context_score": context_score,
            "alpha_score": alpha_score,
            "beta_score": beta_score,
            "probabilistic_score": prob_score,
            "mass_valid": mass_error <= tolerance
        })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        return df

    return df.sort_values("probabilistic_score", ascending=False).head(top_k).reset_index(drop=True)

Cell 20 — Hybrid reranker

In [ ]:
def hybrid_rerank_candidates(
    candidate_df,
    left_context,
    right_context,
    target_mass,
    mass_tolerance=0,
    use_deep=True
):
    if candidate_df is None or len(candidate_df) == 0:
        return pd.DataFrame()

    rows = []

    for _, row in candidate_df.iterrows():
        candidate = row["candidate"]

        ml_score = ml_score_candidate(
            left_context=left_context,
            right_context=right_context,
            candidate=candidate,
            k=11
        )

        if use_deep and HAS_TF:
            deep_score = dl_score_candidate(
                left_context=left_context,
                right_context=right_context,
                candidate=candidate
            )
        else:
            deep_score = 0.0

        candidate_mass = peptide_mass(candidate)
        mass_error = abs(candidate_mass - target_mass)

        mass_valid_score = 1.0 if mass_error <= mass_tolerance else 0.0

        homolog_frequency_score = math.log1p(row["homolog_frequency"])
        context_score_norm = math.log1p(row["context_score"])

        hybrid_score = (
            3.0 * mass_valid_score +
            2.0 * homolog_frequency_score +
            2.0 * context_score_norm +
            1.5 * ml_score +
            1.5 * deep_score -
            2.0 * mass_error
        )

        out = row.to_dict()
        out.update({
            "ml_ensemble_score": ml_score,
            "deep_score": deep_score,
            "mass_valid_score": mass_valid_score,
            "hybrid_score": hybrid_score
        })

        rows.append(out)

    out_df = pd.DataFrame(rows)
    out_df = out_df.sort_values("hybrid_score", ascending=False).reset_index(drop=True)

    return out_df

Cell 21 — Test known-mass CAH2 cases

In [ ]:
cah2_test_cases = [
    {
        "gap_name": "CAH2_gap_1",
        "mass": 420,
        "left": "MSHHWGYGKHN",
        "right": "WHKDFPIAKGER",
        "truth": "GPEH"
    },
    {
        "gap_name": "CAH2_gap_2",
        "mass": 750,
        "left": "DPSLKPL",
        "right": "TSLRILNNGHA",
        "truth": "SVSYDQA"
    },
    {
        "gap_name": "CAH2_gap_3",
        "mass": 622,
        "left": "SQDK",
        "right": "LDGTYRLIQF",
        "truth": "AVLKGGP"
    },
    {
        "gap_name": "CAH2_gap_4",
        "mass": 1318,
        "left": "VHWNT",
        "right": "DGLAVLGIFL",
        "truth": "KYGDFGKAVQQP"
    },
    {
        "gap_name": "CAH2_gap_5",
        "mass": 751,
        "left": "DYWTYPGSL",
        "right": "CVTWIVLKEP",
        "truth": "TTPPLLE"
    },
    {
        "gap_name": "CAH2_gap_6",
        "mass": 544,
        "left": "VSSEQV",
        "right": "KLNFNGEGEP",
        "truth": "LKFR"
    },
    {
        "gap_name": "CAH2_gap_7",
        "mass": 561,
        "left": "PLKNRQI",
        "right": "",
        "truth": "KASFK"
    }
]

known_mass_results = []

for case in cah2_test_cases:
    print("\nRunning", case["gap_name"], "mass:", case["mass"])

    prob_candidates = probabilistic_rank_candidates(
        target_mass=case["mass"],
        left_context=case["left"],
        right_context=case["right"],
        homolog_sequences=cah2_train_seqs,
        tolerance=0,
        top_k=50
    )

    hybrid_candidates = hybrid_rerank_candidates(
        candidate_df=prob_candidates,
        left_context=case["left"],
        right_context=case["right"],
        target_mass=case["mass"],
        mass_tolerance=0,
        use_deep=True
    )

    if len(hybrid_candidates) == 0:
        pred = ""
        rank_of_truth = None
        top_candidate_mass = None
        hybrid_score = None
    else:
        pred = hybrid_candidates.iloc[0]["candidate"]
        top_candidate_mass = hybrid_candidates.iloc[0]["candidate_mass"]
        hybrid_score = hybrid_candidates.iloc[0]["hybrid_score"]

        truth_positions = hybrid_candidates.index[
            hybrid_candidates["candidate"] == case["truth"]
        ].tolist()

        rank_of_truth = truth_positions[0] + 1 if truth_positions else None

    known_mass_results.append({
        "gap_name": case["gap_name"],
        "target_mass": case["mass"],
        "truth": case["truth"],
        "prediction": pred,
        "pred_mass": top_candidate_mass,
        "hybrid_score": hybrid_score,
        "exact_match": pred == case["truth"],
        "rank_of_truth": rank_of_truth,
        "top_5_candidates": hybrid_candidates["candidate"].head(5).tolist() if len(hybrid_candidates) else []
    })

    display(hybrid_candidates.head(10))

known_mass_results_df = pd.DataFrame(known_mass_results)
display(known_mass_results_df)

print("Known-mass exact match accuracy:", known_mass_results_df["exact_match"].mean())
print(
    "Known-mass top-5 recovery:",
    known_mass_results_df["rank_of_truth"].apply(lambda x: x is not None and x <= 5).mean()
)

Cell 22 — Ablation study

In [ ]:
def evaluate_known_mass_ablation(test_cases, homolog_sequences, mass_tolerance=0):
    rows = []

    for case in test_cases:
        prob_candidates = probabilistic_rank_candidates(
            target_mass=case["mass"],
            left_context=case["left"],
            right_context=case["right"],
            homolog_sequences=homolog_sequences,
            tolerance=mass_tolerance,
            top_k=50
        )

        if len(prob_candidates) == 0:
            rows.append({
                "gap_name": case["gap_name"],
                "truth": case["truth"],
                "prob_only_pred": "",
                "hybrid_no_deep_pred": "",
                "hybrid_with_deep_pred": "",
                "prob_only_exact": False,
                "hybrid_no_deep_exact": False,
                "hybrid_with_deep_exact": False
            })
            continue

        prob_only_pred = prob_candidates.iloc[0]["candidate"]

        hybrid_no_deep = hybrid_rerank_candidates(
            candidate_df=prob_candidates,
            left_context=case["left"],
            right_context=case["right"],
            target_mass=case["mass"],
            mass_tolerance=mass_tolerance,
            use_deep=False
        )

        hybrid_with_deep = hybrid_rerank_candidates(
            candidate_df=prob_candidates,
            left_context=case["left"],
            right_context=case["right"],
            target_mass=case["mass"],
            mass_tolerance=mass_tolerance,
            use_deep=True
        )

        hybrid_no_deep_pred = hybrid_no_deep.iloc[0]["candidate"] if len(hybrid_no_deep) else ""
        hybrid_with_deep_pred = hybrid_with_deep.iloc[0]["candidate"] if len(hybrid_with_deep) else ""

        rows.append({
            "gap_name": case["gap_name"],
            "truth": case["truth"],
            "prob_only_pred": prob_only_pred,
            "hybrid_no_deep_pred": hybrid_no_deep_pred,
            "hybrid_with_deep_pred": hybrid_with_deep_pred,
            "prob_only_exact": prob_only_pred == case["truth"],
            "hybrid_no_deep_exact": hybrid_no_deep_pred == case["truth"],
            "hybrid_with_deep_exact": hybrid_with_deep_pred == case["truth"]
        })

    return pd.DataFrame(rows)

ablation_df = evaluate_known_mass_ablation(
    test_cases=cah2_test_cases,
    homolog_sequences=cah2_train_seqs,
    mass_tolerance=0
)

display(ablation_df)

print("Probabilistic-only accuracy:", ablation_df["prob_only_exact"].mean())
print("Hybrid without deep accuracy:", ablation_df["hybrid_no_deep_exact"].mean())
print("Hybrid with deep accuracy:", ablation_df["hybrid_with_deep_exact"].mean())

Cell 23 — Error analysis

In [ ]:
def error_analysis_known_size(eval_df):
    errors = eval_df[eval_df["exact_match"] == False].copy()

    if len(errors) == 0:
        print("No known-size errors found.")
        return pd.DataFrame()

    errors["mass_error"] = (errors["pred_mass"] - errors["truth_mass"]).abs()
    errors["length_error"] = errors["prediction"].str.len() - errors["truth"].str.len()

    summary = {
        "num_errors": len(errors),
        "mean_residue_accuracy_on_errors": errors["residue_accuracy"].mean(),
        "mean_mass_error": errors["mass_error"].mean(),
        "mean_gap_size_error_cases": errors["gap_size"].mean()
    }

    print(summary)

    return errors.sort_values(["residue_accuracy", "mass_error"], ascending=[True, False])

known_size_errors = error_analysis_known_size(known_size_eval_df)
display(known_size_errors.head(20))

Cell 24 — Save all outputs

In [ ]:
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results_df.to_csv(f"{OUTPUT_DIR}/all_ml_model_results.csv", index=False)
top_models_df.to_csv(f"{OUTPUT_DIR}/selected_top_ml_models.csv", index=False)

known_size_eval_df.to_csv(f"{OUTPUT_DIR}/known_size_gap_evaluation.csv", index=False)
known_mass_results_df.to_csv(f"{OUTPUT_DIR}/known_mass_gap_evaluation.csv", index=False)
ablation_df.to_csv(f"{OUTPUT_DIR}/ablation_study.csv", index=False)

if len(known_size_errors) > 0:
    known_size_errors.to_csv(f"{OUTPUT_DIR}/known_size_error_analysis.csv", index=False)

if len(known_mass_errors) > 0:
    known_mass_errors.to_csv(f"{OUTPUT_DIR}/known_mass_error_analysis.csv", index=False)

print("Saved files:")
for file in os.listdir(OUTPUT_DIR):
    if file.endswith(".csv"):
        print(os.path.join(OUTPUT_DIR, file))

In [ ]:
# Cell 25 — Final summary table
summary_rows = []

summary_rows.append({
    "system": "Best single ML model",
    "task": "Known gap size residue prediction",
    "metric": "validation accuracy",
    "score": results_df.iloc[0]["validation_accuracy"]
})

summary_rows.append({
    "system": f"Weighted ML ensemble top {TOP_N_MODELS}",
    "task": "Known gap size residue prediction",
    "metric": "validation accuracy",
    "score": ensemble_val_acc
})

summary_rows.append({
    "system": "ML ensemble iterative gap filling",
    "task": "Known gap size full gap",
    "metric": "exact match accuracy",
    "score": known_size_eval_df["exact_match"].mean()
})

summary_rows.append({
    "system": "ML ensemble iterative gap filling",
    "task": "Known gap size full gap",
    "metric": "mean residue accuracy",
    "score": known_size_eval_df["residue_accuracy"].mean()
})

summary_rows.append({
    "system": "Probabilistic only",
    "task": "Known gap mass",
    "metric": "exact match accuracy",
    "score": ablation_df["prob_only_exact"].mean()
})

summary_rows.append({
    "system": "Hybrid reranker without deep",
    "task": "Known gap mass",
    "metric": "exact match accuracy",
    "score": ablation_df["hybrid_no_deep_exact"].mean()
})

summary_rows.append({
    "system": "Hybrid reranker with deep",
    "task": "Known gap mass",
    "metric": "exact match accuracy",
    "score": ablation_df["hybrid_with_deep_exact"].mean()
})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(f"{OUTPUT_DIR}/final_summary.csv", index=False)

Cell 26 — Inference function for final system

In [ ]:
def final_predict_gap(
    left_context,
    right_context,
    known_gap_size=None,
    known_gap_mass=None,
    homolog_sequences=None,
    mass_tolerance=0,
    top_k=10
):
    """
    Final hybrid inference function.

    Case 1: known_gap_size is provided
        Uses ML ensemble iterative filling.

    Case 2: known_gap_mass is provided
        Uses probabilistic mass candidate generation + hybrid reranking.

    Case 3: both are provided
        Generates mass candidates and can filter/rerank by size too.
    """
    if homolog_sequences is None:
        homolog_sequences = all_homologs

    if known_gap_mass is not None:
        prob_candidates = probabilistic_rank_candidates(
            target_mass=known_gap_mass,
            left_context=left_context,
            right_context=right_context,
            homolog_sequences=homolog_sequences,
            tolerance=mass_tolerance,
            top_k=100
        )

        if known_gap_size is not None and len(prob_candidates) > 0:
            prob_candidates = prob_candidates[
                prob_candidates["candidate_length"] == known_gap_size
            ].reset_index(drop=True)

        hybrid_candidates = hybrid_rerank_candidates(
            candidate_df=prob_candidates,
            left_context=left_context,
            right_context=right_context,
            target_mass=known_gap_mass,
            mass_tolerance=mass_tolerance,
            use_deep=True
        )

        if len(hybrid_candidates) == 0:
            return {
                "mode": "known_mass_hybrid",
                "prediction": "",
                "confidence": 0.0,
                "top_candidates": []
            }

        best = hybrid_candidates.iloc[0]

        return {
            "mode": "known_mass_hybrid",
            "prediction": best["candidate"],
            "confidence": float(best["hybrid_score"]),
            "mass_valid": bool(best["mass_valid"]),
            "pred_mass": int(best["candidate_mass"]),
            "target_mass": int(known_gap_mass),
            "top_candidates": hybrid_candidates.head(top_k).to_dict("records")
        }

    if known_gap_size is not None:
        ml_result = fill_known_size_gap_ml(
            left_context=left_context,
            right_context=right_context,
            gap_size=known_gap_size
        )

        dl_result = dl_predict_gap(left_context, right_context, max_gap=known_gap_size) if HAS_TF else {
            "prediction": "",
            "confidence": 0.0
        }

        return {
            "mode": "known_size_ml",
            "prediction": ml_result["prediction"],
            "confidence": ml_result["confidence"],
            "pred_mass": ml_result["mass"],
            "deep_prediction": dl_result["prediction"],
            "deep_confidence": dl_result["confidence"]
        }

    raise ValueError("You must provide known_gap_size or known_gap_mass.")

# Example 1: known size
example_known_size = final_predict_gap(
    left_context="DIQMTQSPSSL",
    right_context="SASVGDRVTIT",
    known_gap_size=3
)

print("Known-size example:")
print(example_known_size)

# Example 2: known mass
example_known_mass = final_predict_gap(
    left_context="MSHHWGYGKHN",
    right_context="WHKDFPIAKGER",
    known_gap_mass=420,
    homolog_sequences=cah2_train_seqs,
    mass_tolerance=0,
    top_k=5
)

print("\nKnown-mass example:")
print(example_known_mass["prediction"])
print(pd.DataFrame(example_known_mass["top_candidates"]).head())